In [ ]:
# Notebook 12A — Random Forest Regression

## 12A.1 Introduction

This notebook extends the predictive modelling framework established in Notebook 12 by implementing Random Forest Regression for PEMFC stack-voltage prediction.

Notebook 12 previously developed and evaluated Ridge Regression and XGBoost using a leakage-controlled chronological validation framework. Random Forest is introduced here as an additional nonlinear ensemble-learning model so that the final dissertation compares three methodologically distinct modelling approaches:

- Ridge Regression — regularised linear regression;
- Random Forest Regression — nonlinear bagging-based tree ensemble;
- XGBoost Regression — nonlinear gradient-boosting tree ensemble.

The purpose of this notebook is not to redesign the existing modelling framework. Instead, Random Forest is evaluated under the same experimental conditions already established for Ridge and XGBoost so that the subsequent model comparison remains methodologically fair.

Accordingly, this notebook retains the same:

- prediction target: stack voltage;
- final selected predictor set;
- processed feature-engineered dataset;
- development durability stages;
- later-stage holdout stages;
- chronological validation structure;
- inner chronological tuning principle;
- evaluation metrics; and
- leakage-control rules.

Only algorithm-specific modelling and hyperparameter choices are changed where required by the Random Forest learning mechanism.

The principal objective is to determine whether Random Forest can provide competitive and temporally stable prediction of PEMFC stack voltage at unseen later durability stages relative to the previously developed Ridge and XGBoost models.

Following completion of Random Forest development, its predictive performance will be compared with the existing models using the same chronological and later-stage evaluation framework. The better-performing nonlinear tree-based model, Random Forest or XGBoost, will subsequently be selected for post-hoc SHAP interpretation.

In [ ]:
## 12A.2 Broad Workflow

The Random Forest implementation follows the same overall modelling logic established in Notebook 12 so that its results can be compared fairly with Ridge Regression and XGBoost.

The broad workflow of this notebook is:

1. Set up the modelling environment and ensure reproducibility.

2. Define project paths and configure notebook inputs and outputs.

3. Load the same feature-engineered PEMFC dataset used in Notebook 12.

4. Verify dataset integrity, structure, data quality and modelling readiness.

5. Confirm the prediction target, durability-stage identifier and final selected predictor set.

6. Re-establish the leakage-control rules used in Notebook 12.

7. Verify chronological ordering and sequential integrity of the durability-stage data.

8. Reconstruct the development and later-stage holdout datasets:
   - Development stages: 50–850 h
   - Holdout stages: 900, 950 and 1000 h

9. Reconstruct the same outer chronological validation folds used for model evaluation.

10. Reconstruct the chronological inner validation structure used for hyperparameter tuning.

11. Define and evaluate a baseline Random Forest configuration.

12. Define an algorithm-specific Random Forest hyperparameter search space.

13. Perform Random Forest hyperparameter tuning using the same chronological tuning philosophy adopted for XGBoost.

14. Evaluate the tuned Random Forest across the outer chronological validation folds.

15. Compare baseline and tuned Random Forest performance and retain the preferred Random Forest configuration.

16. Fit the final Random Forest model using the complete 50–850 h development dataset.

17. Evaluate the frozen Random Forest model on the untouched later-stage holdout data at 900, 950 and 1000 h.

18. Assess stage-wise predictive performance and residual behaviour.

19. Compare Random Forest with the previously developed Ridge Regression and XGBoost models using the same evaluation criteria.

20. Select the better-performing nonlinear tree-based model, Random Forest or XGBoost, for subsequent SHAP interpretation.

In [ ]:
## 12A.3 Environment Setup and Reproducibility

This section initializes the Python environment required for Random Forest model development and evaluation.

The implementation retains the core data-processing, evaluation and reproducibility libraries used in Notebook 12. Random Forest Regression is imported from scikit-learn for the additional nonlinear bagging-based modelling experiment.

A fixed random seed is maintained throughout the notebook to improve reproducibility of stochastic procedures, including Random Forest construction and subsequent hyperparameter search.

In [1]:
# Core data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt

# System and project utilities
import os
import time
import warnings
import gc
from pathlib import Path

# Random Forest modelling
from sklearn.ensemble import RandomForestRegressor

# Model evaluation
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Hyperparameter search
from sklearn.model_selection import ParameterSampler

# Model persistence
import joblib

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Warning configuration
warnings.filterwarnings("ignore")

print("Environment setup complete.")
print(f"Random seed: {RANDOM_SEED}")

Environment setup complete.
Random seed: 42


In [ ]:
## 12A.4 — Project Paths

Project directories are defined so that the engineered dataset and Random Forest modelling outputs can be accessed and stored consistently within the existing dissertation project structure.

In [2]:
project_root = Path.cwd().parent

data_processed_dir = project_root / "data" / "processed"
results_dir = project_root / "results" / "modeling"
models_dir = project_root / "models"

results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Processed data:", data_processed_dir)
print("Results directory:", results_dir)
print("Models directory:", models_dir)

Project root: C:\Users\usman\Desktop\PEMFC_Dissertation
Processed data: C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed
Results directory: C:\Users\usman\Desktop\PEMFC_Dissertation\results\modeling
Models directory: C:\Users\usman\Desktop\PEMFC_Dissertation\models


In [ ]:
## 12A.5 — Identify the Feature-Engineered Dataset

Before loading the modelling dataset, the processed-data directory is inspected to confirm the availability of the feature-engineered PEMFC dataset used in Notebook 12.

The Random Forest experiment must use the same processed dataset as the previously developed Ridge Regression and XGBoost models to preserve consistency across the final model comparison.

In [3]:
# List available CSV files in the processed-data directory

processed_csv_files = sorted(data_processed_dir.glob("*.csv"))

print("Available processed CSV files:")
print("-" * 40)

for file in processed_csv_files:
    print(file.name)

print("-" * 40)
print(f"Number of CSV files found: {len(processed_csv_files)}")

Available processed CSV files:
----------------------------------------
dataset_structure_summary.csv
ml_variable_classification.csv
operational_cleaned.csv
operational_merged_raw.csv
overall_summary.csv
pemfc_feature_engineered.csv
variable_dictionary.csv
----------------------------------------
Number of CSV files found: 7


In [ ]:
## 12A.6 — Load the Feature-Engineered Dataset

The feature-engineered PEMFC dataset is loaded as the modelling dataset for the Random Forest experiment.

This is the same dataset used for Ridge Regression and XGBoost development in Notebook 12. Reusing the same modelling dataset ensures that differences observed between the models are attributable to their modelling approaches rather than differences in data preparation or feature engineering.

The dataset structure is inspected immediately after loading to confirm its dimensions and available variables before further modelling-readiness checks are performed.

In [4]:
# Define the feature-engineered dataset path

feature_engineered_path = data_processed_dir / "pemfc_feature_engineered.csv"

# Confirm that the expected dataset exists

if not feature_engineered_path.exists():
    raise FileNotFoundError(
        f"Feature-engineered dataset not found: {feature_engineered_path}"
    )

# Load dataset

df = pd.read_csv(feature_engineered_path)

print("Feature-engineered dataset loaded successfully.")
print(f"Dataset shape: {df.shape}")
print(f"Number of observations: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

print("\nDataset columns:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i:>2}. {column}")

Feature-engineered dataset loaded successfully.
Dataset shape: (3629680, 24)
Number of observations: 3,629,680
Number of columns: 24

Dataset columns:
 1. operating_hour
 2. time
 3. current
 4. voltage
 5. power
 6. pressure_anode_inlet
 7. pressure_anode_outlet
 8. pressure_cathode_inlet
 9. pressure_cathode_outlet
10. temp_anode_endplate
11. temp_anode_dewpoint_water
12. temp_anode_inlet
13. temp_anode_outlet
14. temp_cathode_dewpoint_water
15. temp_cathode_inlet
16. temp_cathode_outlet
17. total_anode_stack_flow
18. total_cathode_stack_flow
19. anode_pressure_diff
20. cathode_pressure_diff
21. anode_temp_diff
22. cathode_temp_diff
23. anode_dewpoint_offset
24. cathode_dewpoint_offset


In [ ]:
## 12A.7 — Dataset Integrity, Quality and Modelling Readiness

Before constructing the Random Forest modelling datasets, the loaded feature-engineered dataset is systematically verified for integrity, quality and modelling readiness.

These checks reproduce the essential data-validation principles used in Notebook 12. This ensures that Random Forest receives the same valid modelling data used for the previous Ridge Regression and XGBoost experiments.

### 12A.7.1 — Basic Dataset Structure

The dataset dimensions, column uniqueness and duplicate-row status are examined first to confirm that the loaded modelling dataset has the expected structural integrity.

In [5]:
# Basic structural integrity checks

print("Dataset Structure Verification")
print("-" * 45)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

duplicate_columns = df.columns[df.columns.duplicated()].tolist()
duplicate_rows = df.duplicated().sum()

print(f"\nDuplicate column names: {len(duplicate_columns)}")
print(f"Duplicate rows: {duplicate_rows:,}")

if duplicate_columns:
    print(f"Duplicated columns: {duplicate_columns}")

print("\nStructure check complete.")

Dataset Structure Verification
---------------------------------------------
Rows: 3,629,680
Columns: 24

Duplicate column names: 0
Duplicate rows: 0

Structure check complete.


In [ ]:
### 12A.7.2 — Missing Values, Non-Finite Values and Data Types

The dataset is checked for missing values, non-finite numerical values and inappropriate data types before any modelling subsets are constructed.

Random Forest requires valid numerical predictor and target values. These checks therefore confirm that the loaded feature-engineered dataset remains suitable for direct use within the modelling pipeline and that no additional cleaning has been introduced specifically for the Random Forest experiment.

In [6]:
# Missing-value, non-finite-value and data-type checks

print("Data Quality Verification")
print("-" * 45)

# Missing values
missing_counts = df.isna().sum()
total_missing = int(missing_counts.sum())

print(f"Total missing values: {total_missing:,}")
print(f"Columns containing missing values: {(missing_counts > 0).sum()}")

if total_missing > 0:
    print("\nMissing values by column:")
    print(missing_counts[missing_counts > 0])

# Non-finite values in numerical columns
numeric_columns = df.select_dtypes(include=[np.number]).columns

non_finite_counts = pd.Series(
    {
        column: int((~np.isfinite(df[column].to_numpy())).sum())
        for column in numeric_columns
    }
)

total_non_finite = int(non_finite_counts.sum())

print(f"\nTotal non-finite numerical values: {total_non_finite:,}")
print(
    f"Numerical columns containing non-finite values: "
    f"{(non_finite_counts > 0).sum()}"
)

if total_non_finite > 0:
    print("\nNon-finite values by column:")
    print(non_finite_counts[non_finite_counts > 0])

# Data types
print("\nData types:")
print(df.dtypes)

print("\nData quality check complete.")

Data Quality Verification
---------------------------------------------
Total missing values: 0
Columns containing missing values: 0

Total non-finite numerical values: 0
Numerical columns containing non-finite values: 0

Data types:
operating_hour                   int64
time                           float64
current                        float64
voltage                        float64
power                          float64
pressure_anode_inlet           float64
pressure_anode_outlet          float64
pressure_cathode_inlet         float64
pressure_cathode_outlet        float64
temp_anode_endplate            float64
temp_anode_dewpoint_water      float64
temp_anode_inlet               float64
temp_anode_outlet              float64
temp_cathode_dewpoint_water    float64
temp_cathode_inlet             float64
temp_cathode_outlet            float64
total_anode_stack_flow         float64
total_cathode_stack_flow       float64
anode_pressure_diff            float64
cathode_pressure_diff    

In [ ]:
### 12A.7.3 — Durability-Stage Coverage and Observation Counts

The available durability stages and their observation counts are examined to confirm that the complete chronological sequence required by the modelling framework is present.

The experiment is expected to contain durability stages from 50 h to 1000 h at 50 h intervals. Verifying stage coverage before constructing development, validation and holdout subsets helps ensure that the chronological modelling design is based on the intended experimental sequence.

Observation counts are also examined to identify any unexpected differences in stage size that could affect subsequent model development or evaluation.

In [7]:
# Verify durability-stage coverage and observation counts

expected_stages = list(range(50, 1001, 50))
available_stages = sorted(df["operating_hour"].unique().tolist())

stage_counts = (
    df["operating_hour"]
    .value_counts()
    .sort_index()
)

missing_stages = sorted(set(expected_stages) - set(available_stages))
unexpected_stages = sorted(set(available_stages) - set(expected_stages))

print("Durability-Stage Verification")
print("-" * 45)

print(f"Expected number of stages: {len(expected_stages)}")
print(f"Available number of stages: {len(available_stages)}")

print(f"\nAvailable stages:")
print(available_stages)

print(f"\nMissing expected stages: {missing_stages}")
print(f"Unexpected stages: {unexpected_stages}")

print("\nObservations per durability stage:")
print(stage_counts.to_string())

print(f"\nTotal observations across stages: {stage_counts.sum():,}")

stage_coverage_valid = (
    available_stages == expected_stages
    and len(missing_stages) == 0
    and len(unexpected_stages) == 0
)

print(f"\nComplete expected stage coverage: {stage_coverage_valid}")

Durability-Stage Verification
---------------------------------------------
Expected number of stages: 20
Available number of stages: 20

Available stages:
[50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000]

Missing expected stages: []
Unexpected stages: []

Observations per durability stage:
operating_hour
50      179360
100     179360
150     179360
200     179360
250     179360
300     179360
350     179360
400     179360
450     179360
500     179360
550     179360
600     179360
650     179360
700     179360
750     179360
800     179360
850     179360
900     179360
950     179360
1000    221840

Total observations across stages: 3,629,680

Complete expected stage coverage: True


In [ ]:
### 12A.7.4 — Sample-Level Data Inspection

Representative observations are inspected to confirm that the loaded dataset retains the expected feature structure and durability-stage labels.

Samples from the overall dataset and selected early, intermediate and late durability stages are examined. This provides a direct verification that observations are correctly associated with their operating-hour stages before chronological modelling subsets are constructed.

This step is observational only and does not modify the dataset.

In [8]:
# Inspect representative observations from selected durability stages

inspection_stages = [50, 550, 1000]

for stage in inspection_stages:
    stage_sample = df.loc[
        df["operating_hour"] == stage
    ].head(3)

    print(f"\nRepresentative observations — {stage} h")
    display(stage_sample)


Representative observations — 50 h


,operating_hour,time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,...,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow,anode_pressure_diff,cathode_pressure_diff,anode_temp_diff,cathode_temp_diff,anode_dewpoint_offset,cathode_dewpoint_offset
0,50,1.761,0.0,0.9375,0.0,109.901303,110.32725,109.800128,108.792114,83.128159,...,69.673592,56.337543,0.07,0.291,-0.425947,1.008014,-32.498287,-13.336049,17.486717,5.209305
1,50,2.761,0.0,0.9375,0.0,110.103653,110.32725,109.800128,108.893387,83.078773,...,69.661247,56.324917,0.07,0.291,-0.223597,0.906741,-32.515045,-13.336330,17.437339,5.222213
2,50,3.761,0.0,0.9372,0.0,110.306003,110.32725,109.800128,108.792114,83.078773,...,69.685936,56.299664,0.07,0.291,-0.021247,1.008014,-32.476250,-13.386272,17.360397,5.199318



Representative observations — 550 h


,operating_hour,time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,...,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow,anode_pressure_diff,cathode_pressure_diff,anode_temp_diff,cathode_temp_diff,anode_dewpoint_offset,cathode_dewpoint_offset
1793600,550,1.75,0.0,0.9410,0.0,110.204828,110.529453,109.091903,108.083206,83.639091,...,69.925278,59.921345,0.084,0.349,-0.324625,1.008697,-25.720226,-10.003933,14.147514,5.560028
1793601,550,2.75,0.0,0.9410,0.0,110.103653,110.125047,109.091903,108.083206,83.700821,...,69.863556,59.960102,0.084,0.349,-0.021394,1.008697,-25.783959,-9.903454,14.133423,5.523560
1793602,550,3.75,0.0,0.9407,0.0,109.901303,110.125047,109.091903,108.083206,83.651627,...,69.855614,59.870968,0.084,0.349,-0.223744,1.008697,-25.798152,-9.984646,14.130035,5.479714



Representative observations — 1000 h


,operating_hour,time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,...,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow,anode_pressure_diff,cathode_pressure_diff,anode_temp_diff,cathode_temp_diff,anode_dewpoint_offset,cathode_dewpoint_offset
3407840,1000,1.246,0.0,0.9346,0.0,110.609528,110.832759,110.103653,108.792114,83.546082,...,68.504524,59.230812,0.084,0.349,-0.223231,1.311539,-30.314339,-9.273712,14.307255,5.380169
3407841,1000,2.246,0.0,0.9356,0.0,110.204828,110.832759,109.597778,108.083206,83.607811,...,68.466644,59.192055,0.084,0.349,-0.627931,1.514572,-30.289086,-9.274589,14.269081,5.329368
3407842,1000,3.246,0.0,0.9339,0.0,110.204828,110.832759,109.597778,108.589569,83.632500,...,68.517151,59.204975,0.084,0.349,-0.627931,1.008209,-30.263229,-9.312176,14.217697,5.379875


In [ ]:
### 12A.7.5 — Chronological and Sequential Integrity Verification

The chronological structure of the feature-engineered dataset is verified before constructing the Random Forest development, validation and holdout subsets.

Two levels of temporal organisation are considered:

- durability-stage ordering, represented by `operating_hour`; and
- within-stage measurement ordering, represented by `time`.

The durability stages should follow the expected progression from 50 h to 1000 h. Within each durability stage, measurement time should increase monotonically so that the original sequential organisation of the operational data is preserved.

These checks are particularly important because the modelling framework uses chronological rather than random validation. Any inconsistency in temporal ordering could compromise the intended separation between earlier training observations and later validation or holdout stages.

In [9]:
# Verify chronological durability-stage ordering

observed_stage_sequence = (
    df["operating_hour"]
    .drop_duplicates()
    .tolist()
)

expected_stage_sequence = list(range(50, 1001, 50))

stage_order_valid = observed_stage_sequence == expected_stage_sequence

print("Chronological Integrity Verification")
print("-" * 50)

print("Observed stage sequence:")
print(observed_stage_sequence)

print(f"\nDurability-stage ordering valid: {stage_order_valid}")

Chronological Integrity Verification
--------------------------------------------------
Observed stage sequence:
[50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000]

Durability-stage ordering valid: True


In [10]:
# Verify monotonic time ordering within every durability stage

time_integrity_results = []

for stage, stage_df in df.groupby("operating_hour", sort=True):

    time_values = stage_df["time"]

    time_integrity_results.append({
        "operating_hour": stage,
        "observations": len(stage_df),
        "time_min": time_values.min(),
        "time_max": time_values.max(),
        "monotonic_increasing": time_values.is_monotonic_increasing
    })

time_integrity_df = pd.DataFrame(time_integrity_results)

display(time_integrity_df)

all_stages_monotonic = time_integrity_df["monotonic_increasing"].all()

print(
    f"\nTime monotonically increasing within all durability stages: "
    f"{all_stages_monotonic}"
)

,operating_hour,observations,time_min,time_max,monotonic_increasing
0,50,179360,1.761,179360.761,True
1,100,179360,1.226,179360.226,True
2,150,179360,1.765,179360.765,True
3,200,179360,1.280,179360.280,True
4,250,179360,1.740,179360.740,True
5,300,179360,1.738,179360.738,True
6,350,179360,1.159,179360.159,True
7,400,179360,1.650,179360.650,True
8,450,179360,1.778,179360.778,True
9,500,179360,1.278,179360.278,True



Time monotonically increasing within all durability stages: True


In [ ]:
## 12A.8 — Variable Roles, Target Definition and Leakage Control

Before constructing the Random Forest predictor matrices, the roles of the available variables are explicitly defined.

The Random Forest experiment retains the modelling specification established during the previous feature-selection and model-development stages. No new feature-selection procedure is performed in this notebook, because changing the predictor set specifically for Random Forest would compromise comparability with Ridge Regression and XGBoost.

The following variables require explicit treatment:

- `voltage` — prediction target;
- `operating_hour` — durability-stage identifier used to construct chronological development, validation and holdout subsets;
- `time` — within-stage sequential measurement identifier;
- `power` — excluded from the predictor set because it is mathematically dependent on the prediction target through Power = Voltage × Current.

Neither `operating_hour` nor `time` is used as a Random Forest predictor. They are retained only for chronological organisation, validation and subsequent stage-wise evaluation.

The final Random Forest predictor set will therefore be restricted to the same 20 predictors previously retained for Ridge Regression and XGBoost.

In [6]:
# Define core modelling-variable roles

target_variable = "voltage"
stage_identifier = "operating_hour"
time_identifier = "time"

explicitly_excluded_variables = [
    "power"
]

non_predictor_variables = [
    target_variable,
    stage_identifier,
    time_identifier,
    *explicitly_excluded_variables
]

print("Core Variable Roles")
print("-" * 50)

print(f"Prediction target:        {target_variable}")
print(f"Durability identifier:    {stage_identifier}")
print(f"Within-stage time:        {time_identifier}")
print(f"Explicit leakage control: {explicitly_excluded_variables}")

print("\nVariables not permitted as predictors:")
for variable in non_predictor_variables:
    print(f"  - {variable}")

Core Variable Roles
--------------------------------------------------
Prediction target:        voltage
Durability identifier:    operating_hour
Within-stage time:        time
Explicit leakage control: ['power']

Variables not permitted as predictors:
  - voltage
  - operating_hour
  - time
  - power


In [ ]:
### 12A.8.2 — Verification of Power Exclusion

`power` is excluded from the Random Forest predictor set because electrical power is mathematically derived from the prediction target and current:

Power = Voltage × Current

Including `power` while predicting `voltage` would therefore provide the model with information that directly contains the target variable. This would create target leakage and could produce artificially optimistic predictive performance.

To confirm that this relationship remains present in the loaded feature-engineered dataset, power is reconstructed from the recorded voltage and current measurements and compared with the recorded `power` variable.

This verification reproduces the leakage-control principle established in Notebook 12 and does not modify the dataset.

In [7]:
# Verify mathematical dependency between power, voltage and current

reconstructed_power = df["voltage"] * df["current"]

power_difference = df["power"] - reconstructed_power
absolute_power_difference = power_difference.abs()

print("Power Dependency Verification")
print("-" * 50)

print("Relationship checked:")
print("Power = Voltage × Current")

print(f"\nMean absolute difference: {absolute_power_difference.mean():.10f} W")
print(f"Median absolute difference: {absolute_power_difference.median():.10f} W")
print(f"Maximum absolute difference: {absolute_power_difference.max():.10f} W")

print(
    "\nCorrelation between recorded and reconstructed power: "
    f"{df['power'].corr(reconstructed_power):.10f}"
)

print("\nDecision:")
print(
    "power is excluded from the predictor set because it directly "
    "contains information from the voltage prediction target."
)

Power Dependency Verification
--------------------------------------------------
Relationship checked:
Power = Voltage × Current

Mean absolute difference: 0.0024365677 W
Median absolute difference: 0.0024054800 W
Maximum absolute difference: 0.0067081200 W

Correlation between recorded and reconstructed power: 0.9999998623

Decision:
power is excluded from the predictor set because it directly contains information from the voltage prediction target.


In [ ]:
### 12A.8.3 — Define and Validate the Frozen Predictor Set

Random Forest is evaluated using the same final predictor set previously retained for Ridge Regression and XGBoost.

The predictor set is therefore treated as frozen for the present modelling experiment. Feature selection is not repeated specifically for Random Forest, because changing the input variables between models would reduce the fairness of the subsequent model comparison.

The final predictor set contains 20 operational and engineered variables. Before modelling, the predictor list is checked against the loaded dataset to confirm that:

- all required predictors are present;
- no predictor is duplicated;
- the prediction target is absent from the predictor set;
- chronological identifiers are absent from the predictor set; and
- `power` remains excluded because of its direct mathematical dependence on voltage.

In [8]:
# Frozen 20-predictor set established in the previous feature-selection workflow

predictor_columns = [
    "current",
    "pressure_anode_inlet",
    "pressure_anode_outlet",
    "pressure_cathode_inlet",
    "pressure_cathode_outlet",
    "temp_anode_endplate",
    "temp_anode_dewpoint_water",
    "temp_anode_inlet",
    "temp_anode_outlet",
    "temp_cathode_dewpoint_water",
    "temp_cathode_inlet",
    "temp_cathode_outlet",
    "total_anode_stack_flow",
    "total_cathode_stack_flow",
    "anode_pressure_diff",
    "cathode_pressure_diff",
    "anode_temp_diff",
    "cathode_temp_diff",
    "anode_dewpoint_offset",
    "cathode_dewpoint_offset"
]

# Validation checks
missing_predictors = [
    feature for feature in predictor_columns
    if feature not in df.columns
]

duplicate_predictors = (
    pd.Series(predictor_columns)
    .duplicated()
    .sum()
)

forbidden_predictors = [
    variable for variable in non_predictor_variables
    if variable in predictor_columns
]

print("Frozen Predictor-Set Verification")
print("-" * 50)

print(f"Number of predictors: {len(predictor_columns)}")
print(f"Missing predictors: {missing_predictors}")
print(f"Duplicate predictors: {duplicate_predictors}")
print(f"Forbidden variables included: {forbidden_predictors}")

print("\nFinal predictor set:")
for i, feature in enumerate(predictor_columns, start=1):
    print(f"{i:>2}. {feature}")

predictor_set_valid = (
    len(predictor_columns) == 20
    and len(missing_predictors) == 0
    and duplicate_predictors == 0
    and len(forbidden_predictors) == 0
)

print(f"\nPredictor-set validation passed: {predictor_set_valid}")

Frozen Predictor-Set Verification
--------------------------------------------------
Number of predictors: 20
Missing predictors: []
Duplicate predictors: 0
Forbidden variables included: []

Final predictor set:
 1. current
 2. pressure_anode_inlet
 3. pressure_anode_outlet
 4. pressure_cathode_inlet
 5. pressure_cathode_outlet
 6. temp_anode_endplate
 7. temp_anode_dewpoint_water
 8. temp_anode_inlet
 9. temp_anode_outlet
10. temp_cathode_dewpoint_water
11. temp_cathode_inlet
12. temp_cathode_outlet
13. total_anode_stack_flow
14. total_cathode_stack_flow
15. anode_pressure_diff
16. cathode_pressure_diff
17. anode_temp_diff
18. cathode_temp_diff
19. anode_dewpoint_offset
20. cathode_dewpoint_offset

Predictor-set validation passed: True


In [ ]:
## 12A.9 — Chronological Experimental Design

The Random Forest experiment retains the same chronological development and later-stage holdout structure established in Notebook 12.

The durability stages are divided into two non-overlapping periods:

- Development period: 50–850 h
- Final later-stage holdout: 900, 950 and 1000 h

The development period is used for baseline evaluation, chronological validation and hyperparameter tuning. The later-stage holdout is excluded from all Random Forest model-development decisions and is reserved for final evaluation after the Random Forest configuration has been frozen.

This separation preserves the intended later-stage generalisation test and ensures that Random Forest is evaluated under the same temporal conditions as Ridge Regression and XGBoost.

In [9]:
# Define chronological development and final holdout stages

development_stages = list(range(50, 851, 50))
holdout_stages = [900, 950, 1000]

print("Chronological Experimental Design")
print("-" * 50)

print(f"Development stages ({len(development_stages)}):")
print(development_stages)

print(f"\nFinal holdout stages ({len(holdout_stages)}):")
print(holdout_stages)

Chronological Experimental Design
--------------------------------------------------
Development stages (17):
[50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850]

Final holdout stages (3):
[900, 950, 1000]


In [10]:
# Verify development/holdout separation and stage coverage

development_holdout_overlap = sorted(
    set(development_stages).intersection(holdout_stages)
)

combined_modeling_stages = sorted(
    set(development_stages).union(holdout_stages)
)

all_available_stages = sorted(
    df["operating_hour"].unique().tolist()
)

complete_stage_coverage = (
    combined_modeling_stages == all_available_stages
)

chronological_separation_valid = (
    max(development_stages) < min(holdout_stages)
)

print("Development/Holdout Verification")
print("-" * 50)

print(f"Overlap between development and holdout: {development_holdout_overlap}")
print(f"Chronological separation valid: {chronological_separation_valid}")
print(f"All available stages accounted for: {complete_stage_coverage}")

print(
    f"\nLatest development stage: {max(development_stages)} h"
)
print(
    f"Earliest holdout stage:   {min(holdout_stages)} h"
)

experimental_design_valid = (
    len(development_holdout_overlap) == 0
    and chronological_separation_valid
    and complete_stage_coverage
)

print(
    f"\nExperimental design validation passed: "
    f"{experimental_design_valid}"
)

Development/Holdout Verification
--------------------------------------------------
Overlap between development and holdout: []
Chronological separation valid: True
All available stages accounted for: True

Latest development stage: 850 h
Earliest holdout stage:   900 h

Experimental design validation passed: True


In [ ]:
### 12A.9.3 — Construct Development and Holdout Datasets

The complete feature-engineered dataset is now partitioned according to the validated chronological experimental design.

Observations from 50–850 h form the development dataset. This dataset will support Random Forest baseline evaluation, chronological validation, hyperparameter tuning and final model fitting.

Observations from 900, 950 and 1000 h form the later-stage holdout dataset. These observations remain excluded from Random Forest model development and tuning and will only be used after the final Random Forest configuration has been selected.

The resulting subsets are verified for their dimensions, stage membership and observation counts before predictor and target matrices are constructed.

In [11]:
# Construct chronological development and holdout datasets

development_df = (
    df.loc[df["operating_hour"].isin(development_stages)]
    .copy()
)

holdout_df = (
    df.loc[df["operating_hour"].isin(holdout_stages)]
    .copy()
)

print("Development and Holdout Dataset Construction")
print("-" * 55)

print(f"Development dataset shape: {development_df.shape}")
print(f"Development observations:  {len(development_df):,}")

print(f"\nHoldout dataset shape:      {holdout_df.shape}")
print(f"Holdout observations:       {len(holdout_df):,}")

print("\nDevelopment stages:")
print(sorted(development_df["operating_hour"].unique().tolist()))

print("\nHoldout stages:")
print(sorted(holdout_df["operating_hour"].unique().tolist()))

Development and Holdout Dataset Construction
-------------------------------------------------------
Development dataset shape: (3049120, 24)
Development observations:  3,049,120

Holdout dataset shape:      (580560, 24)
Holdout observations:       580,560

Development stages:
[50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850]

Holdout stages:
[900, 950, 1000]


In [12]:
# Verify that all observations are correctly accounted for

development_count = len(development_df)
holdout_count = len(holdout_df)
combined_count = development_count + holdout_count
original_count = len(df)

print("Observation Accounting")
print("-" * 55)

print(f"Original dataset observations:  {original_count:,}")
print(f"Development observations:       {development_count:,}")
print(f"Holdout observations:           {holdout_count:,}")
print(f"Combined observations:          {combined_count:,}")

print(
    f"\nAll observations accounted for: "
    f"{combined_count == original_count}"
)

print(
    f"Development/holdout index overlap: "
    f"{len(development_df.index.intersection(holdout_df.index))}"
)

Observation Accounting
-------------------------------------------------------
Original dataset observations:  3,629,680
Development observations:       3,049,120
Holdout observations:           580,560
Combined observations:          3,629,680

All observations accounted for: True
Development/holdout index overlap: 0


In [ ]:
### 12A.9.5 — Construct and Validate Predictor and Target Matrices

Predictor and target matrices are constructed separately for the development and later-stage holdout datasets using the frozen modelling specification.

The predictor matrices contain only the 20 previously selected predictors, while `voltage` is retained exclusively as the prediction target.

The durability-stage identifier (`operating_hour`), within-stage sequence variable (`time`) and leakage-related variable (`power`) remain outside the predictor matrices.

No feature standardisation is applied for Random Forest because tree-based split decisions do not depend on the relative numerical scale of the predictors.

The resulting matrices are verified before chronological model development begins.

In [13]:
# Construct Random Forest predictor and target matrices

X_development = development_df[predictor_columns].copy()
y_development = development_df[target_variable].copy()

X_holdout = holdout_df[predictor_columns].copy()
y_holdout = holdout_df[target_variable].copy()

print("Random Forest Modelling Matrices")
print("-" * 55)

print(f"X_development shape: {X_development.shape}")
print(f"y_development shape: {y_development.shape}")

print(f"\nX_holdout shape:     {X_holdout.shape}")
print(f"y_holdout shape:     {y_holdout.shape}")

Random Forest Modelling Matrices
-------------------------------------------------------
X_development shape: (3049120, 20)
y_development shape: (3049120,)

X_holdout shape:     (580560, 20)
y_holdout shape:     (580560,)


In [14]:
# Validate modelling matrices before model development

development_forbidden = [
    variable
    for variable in non_predictor_variables
    if variable in X_development.columns
]

holdout_forbidden = [
    variable
    for variable in non_predictor_variables
    if variable in X_holdout.columns
]

same_predictor_order = (
    X_development.columns.tolist()
    == X_holdout.columns.tolist()
    == predictor_columns
)

matrix_validation_passed = (
    X_development.shape[1] == 20
    and X_holdout.shape[1] == 20
    and len(development_forbidden) == 0
    and len(holdout_forbidden) == 0
    and same_predictor_order
    and len(X_development) == len(y_development)
    and len(X_holdout) == len(y_holdout)
)

print("Modelling Matrix Verification")
print("-" * 55)

print(f"Development predictors: {X_development.shape[1]}")
print(f"Holdout predictors:     {X_holdout.shape[1]}")

print(
    f"\nForbidden variables in development X: "
    f"{development_forbidden}"
)

print(
    f"Forbidden variables in holdout X:     "
    f"{holdout_forbidden}"
)

print(
    f"\nPredictor order identical: "
    f"{same_predictor_order}"
)

print(
    f"Development X/y aligned: "
    f"{len(X_development) == len(y_development)}"
)

print(
    f"Holdout X/y aligned:     "
    f"{len(X_holdout) == len(y_holdout)}"
)

print(
    f"\nModelling matrix validation passed: "
    f"{matrix_validation_passed}"
)

Modelling Matrix Verification
-------------------------------------------------------
Development predictors: 20
Holdout predictors:     20

Forbidden variables in development X: []
Forbidden variables in holdout X:     []

Predictor order identical: True
Development X/y aligned: True
Holdout X/y aligned:     True

Modelling matrix validation passed: True


In [ ]:
## 12A.10 — Outer Chronological Validation Framework

Random Forest is evaluated using the same four expanding chronological outer folds established for Ridge Regression and XGBoost in Notebook 12.

The purpose of the outer validation framework is to evaluate how effectively a model trained on earlier durability stages generalises to subsequently observed durability stages.

Unlike random cross-validation, observations from later durability stages are not allowed to enter the training data for an earlier validation period. This preserves the temporal direction of the durability experiment and provides a more appropriate assessment of later-stage generalisation.

The four outer folds are:

- Fold 1: train on 50–450 h, validate on 500–550 h
- Fold 2: train on 50–550 h, validate on 600–650 h
- Fold 3: train on 50–650 h, validate on 700–750 h
- Fold 4: train on 50–750 h, validate on 800–850 h

These folds are kept identical to the previous modelling framework to ensure that Random Forest, Ridge Regression and XGBoost are evaluated under equivalent chronological conditions.

In [ ]:
## 12A.10 — Outer Chronological Validation Framework

Random Forest is evaluated using the same four expanding chronological outer folds established for Ridge Regression and XGBoost in Notebook 12.

The purpose of the outer validation framework is to evaluate how effectively a model trained on earlier durability stages generalises to subsequently observed durability stages.

Unlike random cross-validation, observations from later durability stages are not allowed to enter the training data for an earlier validation period. This preserves the temporal direction of the durability experiment and provides a more appropriate assessment of later-stage generalisation.

The four outer folds are:

- Fold 1: train on 50–450 h, validate on 500–550 h
- Fold 2: train on 50–550 h, validate on 600–650 h
- Fold 3: train on 50–650 h, validate on 700–750 h
- Fold 4: train on 50–750 h, validate on 800–850 h

These folds are kept identical to the previous modelling framework to ensure that Random Forest, Ridge Regression and XGBoost are evaluated under equivalent chronological conditions.

In [15]:
# Define the same outer chronological folds used in Notebook 12

rf_outer_folds = {
    "Fold_1": {
        "train_labels": list(range(50, 451, 50)),
        "validation_labels": [500, 550]
    },
    "Fold_2": {
        "train_labels": list(range(50, 551, 50)),
        "validation_labels": [600, 650]
    },
    "Fold_3": {
        "train_labels": list(range(50, 651, 50)),
        "validation_labels": [700, 750]
    },
    "Fold_4": {
        "train_labels": list(range(50, 751, 50)),
        "validation_labels": [800, 850]
    }
}

print("Outer Chronological Validation Folds")
print("-" * 65)

for fold_name, fold_info in rf_outer_folds.items():

    print(f"\n{fold_name}")
    print(f"  Training stages:   {fold_info['train_labels']}")
    print(f"  Validation stages: {fold_info['validation_labels']}")

Outer Chronological Validation Folds
-----------------------------------------------------------------

Fold_1
  Training stages:   [50, 100, 150, 200, 250, 300, 350, 400, 450]
  Validation stages: [500, 550]

Fold_2
  Training stages:   [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550]
  Validation stages: [600, 650]

Fold_3
  Training stages:   [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650]
  Validation stages: [700, 750]

Fold_4
  Training stages:   [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750]
  Validation stages: [800, 850]


In [ ]:
### 12A.10.2 — Validate the Outer-Fold Structure

Each outer fold is checked to confirm that:

- training and validation stages do not overlap;
- every validation stage occurs chronologically after the corresponding training stages;
- all stages belong to the development period;
- no final holdout stage is used; and
- the training period expands progressively across folds.

These checks confirm that the Random Forest outer-validation framework preserves the same leakage-controlled chronological structure used for the previous models.

In [16]:
# Validate outer chronological folds

outer_fold_checks = []

for fold_name, fold_info in rf_outer_folds.items():

    train_labels = fold_info["train_labels"]
    validation_labels = fold_info["validation_labels"]

    overlap = sorted(
        set(train_labels).intersection(validation_labels)
    )

    chronological_ordering = (
        max(train_labels) < min(validation_labels)
    )

    all_within_development = (
        set(train_labels + validation_labels)
        .issubset(set(development_stages))
    )

    holdout_excluded = (
        len(
            set(train_labels + validation_labels)
            .intersection(holdout_stages)
        ) == 0
    )

    outer_fold_checks.append({
        "fold": fold_name,
        "train_start_h": min(train_labels),
        "train_end_h": max(train_labels),
        "n_train_stages": len(train_labels),
        "validation_start_h": min(validation_labels),
        "validation_end_h": max(validation_labels),
        "n_validation_stages": len(validation_labels),
        "overlap": len(overlap),
        "chronological_ordering": chronological_ordering,
        "within_development": all_within_development,
        "holdout_excluded": holdout_excluded
    })

outer_fold_check_df = pd.DataFrame(outer_fold_checks)

display(outer_fold_check_df)

outer_framework_valid = (
    (outer_fold_check_df["overlap"] == 0).all()
    and outer_fold_check_df["chronological_ordering"].all()
    and outer_fold_check_df["within_development"].all()
    and outer_fold_check_df["holdout_excluded"].all()
)

print(
    f"\nOuter chronological framework validation passed: "
    f"{outer_framework_valid}"
)

,fold,train_start_h,train_end_h,n_train_stages,validation_start_h,validation_end_h,n_validation_stages,overlap,chronological_ordering,within_development,holdout_excluded
0,Fold_1,50,450,9,500,550,2,0,True,True,True
1,Fold_2,50,550,11,600,650,2,0,True,True,True
2,Fold_3,50,650,13,700,750,2,0,True,True,True
3,Fold_4,50,750,15,800,850,2,0,True,True,True



Outer chronological framework validation passed: True


In [ ]:
## 12A.11 — Inner Chronological Tuning Framework

Random Forest hyperparameters are tuned using chronological validation performed entirely within each outer training period.

The outer validation stages are reserved for evaluating the Random Forest configuration selected through inner validation and are therefore not used for hyperparameter selection.

To maintain methodological comparability with the XGBoost tuning procedure established in Notebook 12, valid expanding inner chronological splits are first generated within each outer training fold. Each inner split uses earlier durability stages for training and the subsequent two durability stages for validation.

Because exhaustive tuning across every possible inner split would substantially increase computational cost for the multi-million-observation dataset, the same computational strategy used for XGBoost is retained: the two most recent valid inner chronological splits within each outer training fold are used for formal hyperparameter tuning.

This maintains repeated chronological validation while concentrating tuning assessment on the later portions of each available outer-training period.

In [17]:
# Generate all valid expanding inner chronological splits
# entirely within each outer-training period

rf_all_inner_splits = {}

for fold_name, fold_info in rf_outer_folds.items():

    outer_train_labels = fold_info["train_labels"]

    fold_inner_splits = []

    # Require 5 initial training stages.
    # Each subsequent inner validation block contains 2 stages.
    for validation_start_idx in range(
        5,
        len(outer_train_labels) - 1,
        2
    ):

        inner_train_labels = outer_train_labels[:validation_start_idx]

        inner_validation_labels = outer_train_labels[
            validation_start_idx:
            validation_start_idx + 2
        ]

        if len(inner_validation_labels) == 2:

            fold_inner_splits.append({
                "train_labels": inner_train_labels,
                "validation_labels": inner_validation_labels
            })

    rf_all_inner_splits[fold_name] = fold_inner_splits


print("All Valid Inner Chronological Splits")
print("-" * 70)

for fold_name, splits in rf_all_inner_splits.items():

    print(f"\n{fold_name}: {len(splits)} valid inner split(s)")

    for split_number, split_info in enumerate(splits, start=1):

        print(
            f"  Inner Split {split_number}: "
            f"Train {split_info['train_labels']} "
            f"-> Validate {split_info['validation_labels']}"
        )

All Valid Inner Chronological Splits
----------------------------------------------------------------------

Fold_1: 2 valid inner split(s)
  Inner Split 1: Train [50, 100, 150, 200, 250] -> Validate [300, 350]
  Inner Split 2: Train [50, 100, 150, 200, 250, 300, 350] -> Validate [400, 450]

Fold_2: 3 valid inner split(s)
  Inner Split 1: Train [50, 100, 150, 200, 250] -> Validate [300, 350]
  Inner Split 2: Train [50, 100, 150, 200, 250, 300, 350] -> Validate [400, 450]
  Inner Split 3: Train [50, 100, 150, 200, 250, 300, 350, 400, 450] -> Validate [500, 550]

Fold_3: 4 valid inner split(s)
  Inner Split 1: Train [50, 100, 150, 200, 250] -> Validate [300, 350]
  Inner Split 2: Train [50, 100, 150, 200, 250, 300, 350] -> Validate [400, 450]
  Inner Split 3: Train [50, 100, 150, 200, 250, 300, 350, 400, 450] -> Validate [500, 550]
  Inner Split 4: Train [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550] -> Validate [600, 650]

Fold_4: 5 valid inner split(s)
  Inner Split 1: Train [5

In [ ]:
### 12A.11.2 — Select Inner Splits for Formal Random Forest Tuning

From the valid inner chronological splits generated within each outer fold, the two most recent splits are retained for formal Random Forest hyperparameter tuning.

This reproduces the computationally controlled tuning strategy used for XGBoost. It avoids evaluating every hyperparameter configuration across all possible inner splits while preserving repeated chronological assessment.

No outer-validation or final-holdout observations are used during this tuning process.

In [18]:
# Retain the two most recent valid inner splits
# within each outer-training fold

rf_inner_splits = {}

for fold_name, splits in rf_all_inner_splits.items():

    if len(splits) < 2:
        raise ValueError(
            f"{fold_name} contains fewer than two valid inner splits."
        )

    rf_inner_splits[fold_name] = splits[-2:]


print("Selected Inner Splits for Formal RF Tuning")
print("-" * 70)

for fold_name, splits in rf_inner_splits.items():

    print(f"\n{fold_name}")

    for split_number, split_info in enumerate(splits, start=1):

        print(
            f"  Selected Inner Split {split_number}: "
            f"Train {split_info['train_labels']} "
            f"-> Validate {split_info['validation_labels']}"
        )

Selected Inner Splits for Formal RF Tuning
----------------------------------------------------------------------

Fold_1
  Selected Inner Split 1: Train [50, 100, 150, 200, 250] -> Validate [300, 350]
  Selected Inner Split 2: Train [50, 100, 150, 200, 250, 300, 350] -> Validate [400, 450]

Fold_2
  Selected Inner Split 1: Train [50, 100, 150, 200, 250, 300, 350] -> Validate [400, 450]
  Selected Inner Split 2: Train [50, 100, 150, 200, 250, 300, 350, 400, 450] -> Validate [500, 550]

Fold_3
  Selected Inner Split 1: Train [50, 100, 150, 200, 250, 300, 350, 400, 450] -> Validate [500, 550]
  Selected Inner Split 2: Train [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550] -> Validate [600, 650]

Fold_4
  Selected Inner Split 1: Train [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550] -> Validate [600, 650]
  Selected Inner Split 2: Train [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650] -> Validate [700, 750]


In [ ]:
### 12A.11.3 — Validate the Inner Chronological Tuning Framework

The selected inner splits are formally checked before Random Forest tuning begins.

For each inner split, the following conditions must hold:

- inner-training and inner-validation stages do not overlap;
- inner-validation stages occur strictly after the corresponding inner-training stages;
- all inner-training and inner-validation stages remain within the corresponding outer-training period;
- outer-validation stages are excluded from inner tuning; and
- final holdout stages (900–1000 h) remain completely excluded.

This establishes the nested chronological hierarchy required for leakage-controlled hyperparameter selection:

Inner training → Inner validation → Outer validation → Final holdout.

In [19]:
# Validate selected inner chronological tuning splits

inner_validation_checks = []

for fold_name, selected_splits in rf_inner_splits.items():

    outer_train_labels = rf_outer_folds[fold_name]["train_labels"]
    outer_validation_labels = rf_outer_folds[fold_name]["validation_labels"]

    for split_number, split_info in enumerate(selected_splits, start=1):

        inner_train = split_info["train_labels"]
        inner_validation = split_info["validation_labels"]

        inner_overlap = len(
            set(inner_train).intersection(inner_validation)
        )

        chronological_ordering = (
            max(inner_train) < min(inner_validation)
        )

        contained_in_outer_train = (
            set(inner_train + inner_validation)
            .issubset(set(outer_train_labels))
        )

        outer_validation_excluded = (
            len(
                set(inner_train + inner_validation)
                .intersection(outer_validation_labels)
            ) == 0
        )

        final_holdout_excluded = (
            len(
                set(inner_train + inner_validation)
                .intersection(holdout_stages)
            ) == 0
        )

        inner_before_outer_validation = (
            max(inner_validation) < min(outer_validation_labels)
        )

        inner_validation_checks.append({
            "fold": fold_name,
            "inner_split": split_number,
            "inner_train_end_h": max(inner_train),
            "inner_validation_start_h": min(inner_validation),
            "inner_validation_end_h": max(inner_validation),
            "outer_validation_start_h": min(outer_validation_labels),
            "inner_overlap": inner_overlap,
            "chronological_ordering": chronological_ordering,
            "contained_in_outer_train": contained_in_outer_train,
            "outer_validation_excluded": outer_validation_excluded,
            "inner_before_outer_validation": inner_before_outer_validation,
            "final_holdout_excluded": final_holdout_excluded
        })

inner_validation_check_df = pd.DataFrame(inner_validation_checks)

display(inner_validation_check_df)

inner_framework_valid = (
    (inner_validation_check_df["inner_overlap"] == 0).all()
    and inner_validation_check_df["chronological_ordering"].all()
    and inner_validation_check_df["contained_in_outer_train"].all()
    and inner_validation_check_df["outer_validation_excluded"].all()
    and inner_validation_check_df["inner_before_outer_validation"].all()
    and inner_validation_check_df["final_holdout_excluded"].all()
)

print(
    f"\nInner chronological tuning framework validation passed: "
    f"{inner_framework_valid}"
)

,fold,inner_split,inner_train_end_h,inner_validation_start_h,inner_validation_end_h,outer_validation_start_h,inner_overlap,chronological_ordering,contained_in_outer_train,outer_validation_excluded,inner_before_outer_validation,final_holdout_excluded
0,Fold_1,1,250,300,350,500,0,True,True,True,True,True
1,Fold_1,2,350,400,450,500,0,True,True,True,True,True
2,Fold_2,1,350,400,450,600,0,True,True,True,True,True
3,Fold_2,2,450,500,550,600,0,True,True,True,True,True
4,Fold_3,1,450,500,550,700,0,True,True,True,True,True
5,Fold_3,2,550,600,650,700,0,True,True,True,True,True
6,Fold_4,1,550,600,650,800,0,True,True,True,True,True
7,Fold_4,2,650,700,750,800,0,True,True,True,True,True



Inner chronological tuning framework validation passed: True


In [ ]:
### 12A.12 — Verification of Observation-Level Equivalence with XGBoost

Before Random Forest model fitting, the chronological data subsets are
verified against the experimental structure used for XGBoost in Notebook 12.

Random Forest uses the same `development_df`, the same durability-stage
membership rules and all observations belonging to each selected stage.

No observation-level subsampling is introduced.

This ensures that differences between Random Forest and XGBoost performance
cannot arise from differences in the number or chronological location of
training and validation observations.

In [20]:
# Expected observation counts from the original XGBoost implementation

xgb_outer_expected_counts = {
    "Fold_1": {
        "train_observations": 1614240,
        "validation_observations": 358720
    },
    "Fold_2": {
        "train_observations": 1972960,
        "validation_observations": 358720
    },
    "Fold_3": {
        "train_observations": 2331680,
        "validation_observations": 358720
    },
    "Fold_4": {
        "train_observations": 2690400,
        "validation_observations": 358720
    }
}

rf_outer_count_records = []

for fold_name, fold_info in rf_outer_folds.items():

    train_mask = development_df["operating_hour"].isin(
        fold_info["train_labels"]
    )

    validation_mask = development_df["operating_hour"].isin(
        fold_info["validation_labels"]
    )

    rf_train_count = int(train_mask.sum())
    rf_validation_count = int(validation_mask.sum())

    expected_train = (
        xgb_outer_expected_counts[fold_name]["train_observations"]
    )

    expected_validation = (
        xgb_outer_expected_counts[fold_name]["validation_observations"]
    )

    rf_outer_count_records.append({
        "Fold": fold_name,
        "RF_Train_Observations": rf_train_count,
        "XGB_Train_Observations": expected_train,
        "Train_Count_Match": rf_train_count == expected_train,
        "RF_Validation_Observations": rf_validation_count,
        "XGB_Validation_Observations": expected_validation,
        "Validation_Count_Match":
            rf_validation_count == expected_validation
    })

rf_xgb_outer_equivalence_df = pd.DataFrame(
    rf_outer_count_records
)

display(rf_xgb_outer_equivalence_df)

outer_observation_equivalence = (
    rf_xgb_outer_equivalence_df["Train_Count_Match"].all()
    and
    rf_xgb_outer_equivalence_df["Validation_Count_Match"].all()
)

print(
    "\nRF/XGBoost outer-fold observation equivalence passed:",
    outer_observation_equivalence
)

,Fold,RF_Train_Observations,XGB_Train_Observations,Train_Count_Match,RF_Validation_Observations,XGB_Validation_Observations,Validation_Count_Match
0,Fold_1,1614240,1614240,True,358720,358720,True
1,Fold_2,1972960,1972960,True,358720,358720,True
2,Fold_3,2331680,2331680,True,358720,358720,True
3,Fold_4,2690400,2690400,True,358720,358720,True



RF/XGBoost outer-fold observation equivalence passed: True


In [21]:
# Verify observation counts for the RF inner tuning splits

rf_inner_count_records = []

for fold_name, selected_splits in rf_inner_splits.items():

    for split_number, split_info in enumerate(
        selected_splits,
        start=1
    ):

        inner_train_mask = development_df[
            "operating_hour"
        ].isin(
            split_info["train_labels"]
        )

        inner_validation_mask = development_df[
            "operating_hour"
        ].isin(
            split_info["validation_labels"]
        )

        train_count = int(inner_train_mask.sum())
        validation_count = int(inner_validation_mask.sum())

        rf_inner_count_records.append({
            "Fold": fold_name,
            "Inner_Split": split_number,
            "Train_End_h": max(
                split_info["train_labels"]
            ),
            "Validation_Hours": str(
                split_info["validation_labels"]
            ),
            "Train_Observations": train_count,
            "Validation_Observations": validation_count
        })

rf_inner_count_df = pd.DataFrame(
    rf_inner_count_records
)

display(rf_inner_count_df)

,Fold,Inner_Split,Train_End_h,Validation_Hours,Train_Observations,Validation_Observations
0,Fold_1,1,250,"[300, 350]",896800,358720
1,Fold_1,2,350,"[400, 450]",1255520,358720
2,Fold_2,1,350,"[400, 450]",1255520,358720
3,Fold_2,2,450,"[500, 550]",1614240,358720
4,Fold_3,1,450,"[500, 550]",1614240,358720
5,Fold_3,2,550,"[600, 650]",1972960,358720
6,Fold_4,1,550,"[600, 650]",1972960,358720
7,Fold_4,2,650,"[700, 750]",2331680,358720


In [ ]:
## 12A.13 — Baseline Random Forest Model

A baseline Random Forest regression model is established before hyperparameter tuning.

The purpose of this model is to provide a genuinely untuned reference against which the benefit of subsequent chronological hyperparameter tuning can be assessed. Dataset-specific values are therefore not selected in advance based on anticipated or observed predictive performance.

The principal statistical hyperparameters follow the standard RandomForestRegressor configuration:

- 100 trees (`n_estimators=100`);
- squared-error split criterion;
- unrestricted maximum tree depth;
- minimum split size of 2 observations;
- minimum terminal leaf size of 1 observation;
- all predictors available as candidates at each split (`max_features=1.0`);
- bootstrap sampling enabled; and
- full-size bootstrap samples (`max_samples=None`).

Two implementation controls are specified separately. `random_state=42` is used to ensure reproducibility, while `n_jobs=-1` enables parallel tree construction using the available CPU resources. Neither setting represents predictive hyperparameter optimisation.

Although unrestricted tree growth and small terminal leaves may be computationally demanding for the present multi-million-observation dataset, these characteristics are retained in the baseline rather than being modified a priori. Their suitability will subsequently be investigated through the predefined chronological tuning framework.

The baseline is evaluated using the same four outer chronological folds and the same observations previously used for XGBoost, ensuring data-level comparability between the models.

In [22]:
# Define the untuned baseline Random Forest configuration

rf_baseline_config = {
    "n_estimators": 100,
    "criterion": "squared_error",
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "bootstrap": True,
    "max_samples": None,
    "random_state": RANDOM_SEED,
    "n_jobs": -1
}

print("Baseline Random Forest Configuration")
print("-" * 55)

for parameter, value in rf_baseline_config.items():
    print(f"{parameter}: {value}")

Baseline Random Forest Configuration
-------------------------------------------------------
n_estimators: 100
criterion: squared_error
max_depth: None
min_samples_split: 2
min_samples_leaf: 1
max_features: 1.0
bootstrap: True
max_samples: None
random_state: 42
n_jobs: -1


In [ ]:
### 12A.13.2 — Baseline Random Forest Evaluation: Fold 1

The predefined baseline Random Forest is first evaluated on the first outer chronological fold.

Fold 1 trains the model using durability stages from 50 to 450 h and evaluates generalisation on the subsequently observed 500 and 550 h stages.

The training and validation observations are identical to those used for the corresponding XGBoost experiment in Notebook 12.

Training time, prediction time and the predefined regression metrics (RMSE, MAE and R²) are recorded. Basic tree-complexity diagnostics are also inspected after fitting because the baseline allows unrestricted tree growth.

The final 900–1000 h holdout remains completely untouched.

In [28]:
# Baseline Random Forest — Outer Fold 1

fold_name = "Fold_1"

train_labels = rf_outer_folds[fold_name]["train_labels"]
validation_labels = rf_outer_folds[fold_name]["validation_labels"]

# --------------------------------------------------
# Construct the exact chronological masks
# --------------------------------------------------

train_mask = development_df["operating_hour"].isin(train_labels)
validation_mask = development_df["operating_hour"].isin(validation_labels)

X_train_fold1 = development_df.loc[train_mask, predictor_columns]
y_train_fold1 = development_df.loc[train_mask, target_variable]

X_validation_fold1 = development_df.loc[
    validation_mask,
    predictor_columns
]
y_validation_fold1 = development_df.loc[
    validation_mask,
    target_variable
]

print("Baseline Random Forest — Fold 1")
print("-" * 60)

print(f"Training stages:   {train_labels}")
print(f"Validation stages: {validation_labels}")

print(f"\nTraining observations:   {len(X_train_fold1):,}")
print(f"Validation observations: {len(X_validation_fold1):,}")

# --------------------------------------------------
# Initialize predefined baseline model
# --------------------------------------------------

rf_baseline_fold1 = RandomForestRegressor(
    **rf_baseline_config
)

# --------------------------------------------------
# Fit model
# --------------------------------------------------

fit_start = time.perf_counter()

rf_baseline_fold1.fit(
    X_train_fold1,
    y_train_fold1
)

fit_end = time.perf_counter()

training_time_seconds = fit_end - fit_start

# --------------------------------------------------
# Predict on outer validation data
# --------------------------------------------------

prediction_start = time.perf_counter()

y_validation_pred_fold1 = rf_baseline_fold1.predict(
    X_validation_fold1
)

prediction_end = time.perf_counter()

prediction_time_seconds = (
    prediction_end - prediction_start
)

# --------------------------------------------------
# Calculate performance metrics
# --------------------------------------------------

rmse_fold1 = np.sqrt(
    mean_squared_error(
        y_validation_fold1,
        y_validation_pred_fold1
    )
)

mae_fold1 = mean_absolute_error(
    y_validation_fold1,
    y_validation_pred_fold1
)

r2_fold1 = r2_score(
    y_validation_fold1,
    y_validation_pred_fold1
)

# --------------------------------------------------
# Tree-complexity diagnostics
# --------------------------------------------------

tree_depths_fold1 = np.array([
    estimator.tree_.max_depth
    for estimator in rf_baseline_fold1.estimators_
])

tree_nodes_fold1 = np.array([
    estimator.tree_.node_count
    for estimator in rf_baseline_fold1.estimators_
])

# --------------------------------------------------
# Display results
# --------------------------------------------------

print("\nFold 1 Performance")
print("-" * 60)

print(f"RMSE: {rmse_fold1:.6f} V")
print(f"MAE:  {mae_fold1:.6f} V")
print(f"R²:   {r2_fold1:.6f}")

print("\nRuntime")
print("-" * 60)

print(
    f"Training time:   "
    f"{training_time_seconds:.2f} seconds "
    f"({training_time_seconds / 60:.2f} minutes)"
)

print(
    f"Prediction time: "
    f"{prediction_time_seconds:.2f} seconds "
    f"({prediction_time_seconds / 60:.2f} minutes)"
)

print("\nTree Complexity")
print("-" * 60)

print(
    f"Mean tree depth:   "
    f"{tree_depths_fold1.mean():.2f}"
)

print(
    f"Minimum depth:     "
    f"{tree_depths_fold1.min()}"
)

print(
    f"Maximum depth:     "
    f"{tree_depths_fold1.max()}"
)

print(
    f"Mean node count:   "
    f"{tree_nodes_fold1.mean():,.0f}"
)

print(
    f"Minimum nodes:     "
    f"{tree_nodes_fold1.min():,}"
)

print(
    f"Maximum nodes:     "
    f"{tree_nodes_fold1.max():,}"
)

Baseline Random Forest — Fold 1
------------------------------------------------------------
Training stages:   [50, 100, 150, 200, 250, 300, 350, 400, 450]
Validation stages: [500, 550]

Training observations:   1,614,240
Validation observations: 358,720


MemoryError: could not allocate 134217728 bytes

In [28]:
# --------------------------------------------------
# Baseline Random Forest — memory-controlled execution
# --------------------------------------------------
# IMPORTANT:
# Only n_jobs is changed from -1 to 1.
# The statistical RF specification remains unchanged.

rf_baseline_config_sequential = rf_baseline_config.copy()
rf_baseline_config_sequential["n_jobs"] = 1

print("Baseline Random Forest — Sequential Execution")
print("-" * 60)

for parameter, value in rf_baseline_config_sequential.items():
    print(f"{parameter}: {value}")

Baseline Random Forest — Sequential Execution
------------------------------------------------------------
n_estimators: 100
criterion: squared_error
max_depth: None
min_samples_split: 2
min_samples_leaf: 1
max_features: 1.0
bootstrap: True
max_samples: None
random_state: 42
n_jobs: 1


In [29]:
# --------------------------------------------------
# Computational feasibility probe
# --------------------------------------------------
# This is NOT a predictive baseline result.
# It tests whether one unrestricted RF tree can be
# constructed successfully on the full Fold 1 training set.
# --------------------------------------------------

fold_name = "Fold_1"

train_labels = rf_outer_folds[fold_name]["train_labels"]

train_mask = development_df["operating_hour"].isin(train_labels)

X_train_fold1 = development_df.loc[
    train_mask,
    predictor_columns
]

y_train_fold1 = development_df.loc[
    train_mask,
    target_variable
]

print("Single-Tree Computational Feasibility Probe")
print("-" * 60)
print(f"Training stages: {train_labels}")
print(f"Training observations: {len(X_train_fold1):,}")

rf_feasibility_probe = RandomForestRegressor(
    n_estimators=1,
    criterion="squared_error",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=1.0,
    bootstrap=True,
    max_samples=None,
    random_state=RANDOM_SEED,
    n_jobs=1
)

probe_start = time.perf_counter()

rf_feasibility_probe.fit(
    X_train_fold1,
    y_train_fold1
)

probe_time = time.perf_counter() - probe_start

probe_tree = rf_feasibility_probe.estimators_[0]

print("\nProbe completed successfully.")
print(f"Training time: {probe_time:.2f} seconds")
print(f"Training time: {probe_time / 60:.2f} minutes")
print(f"Tree depth: {probe_tree.tree_.max_depth}")
print(f"Node count: {probe_tree.tree_.node_count:,}")

Single-Tree Computational Feasibility Probe
------------------------------------------------------------
Training stages: [50, 100, 150, 200, 250, 300, 350, 400, 450]
Training observations: 1,614,240

Probe completed successfully.
Training time: 72.59 seconds
Training time: 1.21 minutes
Tree depth: 62
Node count: 1,407,579


In [ ]:
### 12A.13.3 — Computational Feasibility Assessment and Baseline Adaptation

The standard untuned Random Forest baseline could not be fitted successfully on the first outer chronological fold under parallel execution.

Fold 1 contained 1,614,240 training observations and 20 predictors. The baseline used 100 trees, unrestricted tree depth (`max_depth=None`), a minimum leaf size of one observation (`min_samples_leaf=1`), full predictor availability at each split (`max_features=1.0`) and bootstrap sampling.

During model fitting with `n_jobs=-1`, execution terminated with a `MemoryError` while scikit-learn was expanding the decision trees. This indicated that the standard unrestricted Random Forest configuration exceeded the available memory under parallel tree construction.

To distinguish parallelisation-related memory pressure from the intrinsic complexity of the unrestricted trees, a computational feasibility probe was subsequently performed. A single tree was fitted to the complete Fold 1 training set using the same statistical tree-growth settings but with sequential execution (`n_jobs=1`).

The single tree fitted successfully but reached:

- maximum depth: 62;
- node count: 1,407,579; and
- fitting time: 72.59 seconds.

These results demonstrate that unrestricted tree growth produces extremely large individual trees for the present high-frequency PEMFC dataset. Although sequential execution reduces the number of trees being constructed simultaneously, it does not remove the memory required to retain the completed trees within the forest. Consequently, simply changing `n_jobs` does not address the underlying tree-complexity problem.

The standard baseline attempt is therefore retained as a computational-feasibility finding rather than being silently replaced. A computationally constrained Random Forest baseline will next be defined by introducing explicit tree-complexity control while retaining the complete chronological training data, the same 20 predictors, the same target variable and the same outer validation structure used for XGBoost.

The constrained baseline parameters will be selected from methodological evidence and the observed computational behaviour rather than from validation-set predictive performance. Subsequent hyperparameter optimisation will remain confined to the predefined inner chronological validation framework.

In [ ]:
### 12A.13.4 — Training-Only Tree-Depth Feasibility Assessment

The unrestricted Random Forest feasibility probe demonstrated that a single
tree fitted to the complete Fold 1 training data reached a depth of 62 and
contained 1,407,579 nodes. At forest scale, unrestricted parallel tree
construction exceeded the available system memory.

Random Forest methodology traditionally permits extensively grown individual
trees. However, implementation guidance for RandomForestRegressor explicitly
notes that unrestricted tree-growth settings can generate very large trees and
that tree complexity may be controlled to reduce memory consumption.

Because the literature does not prescribe a universally optimal maximum depth
for the present PEMFC regression problem, a training-only computational
assessment is performed before defining a feasible baseline.

Candidate maximum depths of 10, 15, 20, 25 and 30 are examined using a single
tree fitted to the complete Fold 1 training dataset. All other statistical
settings are held constant.

This assessment does not use the outer validation stages and does not calculate
predictive performance metrics. Candidate depths are compared only in terms of
tree size and computational behaviour. Consequently, the assessment is used
to establish computational feasibility rather than predictive hyperparameter
optimisation.

The objective is to identify the least restrictive depth constraint that
substantially reduces the extreme tree complexity observed under unrestricted
growth while preserving the complete training dataset and reserving predictive
hyperparameter selection for the subsequent inner chronological tuning stage.

In [33]:
# --------------------------------------------------
# Training-only tree-depth feasibility assessment
# --------------------------------------------------

depth_candidates = [10, 15, 20, 25, 30]

depth_feasibility_results = []

print("Training-Only Tree-Depth Feasibility Assessment")
print("=" * 70)

print(f"Training observations: {len(X_train_fold1):,}")
print(f"Predictors: {X_train_fold1.shape[1]}")
print("Validation data used: No")
print()

for depth in depth_candidates:

    print(f"Testing max_depth = {depth} ...")

    probe_model = RandomForestRegressor(
        n_estimators=1,
        criterion="squared_error",
        max_depth=depth,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=1.0,
        bootstrap=True,
        max_samples=None,
        random_state=RANDOM_SEED,
        n_jobs=1
    )

    start_time = time.perf_counter()

    probe_model.fit(
        X_train_fold1,
        y_train_fold1
    )

    elapsed_time = time.perf_counter() - start_time

    probe_tree = probe_model.estimators_[0]

    achieved_depth = probe_tree.tree_.max_depth
    node_count = probe_tree.tree_.node_count
    leaf_count = probe_tree.tree_.n_leaves

    node_reduction_pct = (
        1 -
        node_count / 1407579
    ) * 100

    depth_feasibility_results.append({
        "max_depth_setting": depth,
        "achieved_depth": achieved_depth,
        "node_count": node_count,
        "leaf_count": leaf_count,
        "training_time_seconds": elapsed_time,
        "node_reduction_vs_unrestricted_pct":
            node_reduction_pct
    })

    print(
        f"  Achieved depth: {achieved_depth}"
    )

    print(
        f"  Nodes: {node_count:,}"
    )

    print(
        f"  Leaves: {leaf_count:,}"
    )

    print(
        f"  Runtime: {elapsed_time:.2f} seconds"
    )

    print(
        f"  Node reduction vs unrestricted: "
        f"{node_reduction_pct:.2f}%"
    )

    print()

    # Release model before fitting next candidate
    del probe_model
    gc.collect()


depth_feasibility_df = pd.DataFrame(
    depth_feasibility_results
)

print("\nDepth Feasibility Summary")
print("=" * 70)

display(depth_feasibility_df)

Training-Only Tree-Depth Feasibility Assessment
Training observations: 1,614,240
Predictors: 20
Validation data used: No

Testing max_depth = 10 ...
  Achieved depth: 10
  Nodes: 1,909
  Leaves: 955
  Runtime: 22.77 seconds
  Node reduction vs unrestricted: 99.86%

Testing max_depth = 15 ...
  Achieved depth: 15
  Nodes: 33,091
  Leaves: 16,546
  Runtime: 42.45 seconds
  Node reduction vs unrestricted: 97.65%

Testing max_depth = 20 ...
  Achieved depth: 20
  Nodes: 207,771
  Leaves: 103,886
  Runtime: 51.66 seconds
  Node reduction vs unrestricted: 85.24%

Testing max_depth = 25 ...
  Achieved depth: 25
  Nodes: 614,855
  Leaves: 307,428
  Runtime: 60.67 seconds
  Node reduction vs unrestricted: 56.32%

Testing max_depth = 30 ...
  Achieved depth: 30
  Nodes: 1,043,755
  Leaves: 521,878
  Runtime: 65.26 seconds
  Node reduction vs unrestricted: 25.85%


Depth Feasibility Summary


,max_depth_setting,achieved_depth,node_count,leaf_count,training_time_seconds,node_reduction_vs_unrestricted_pct
0,10,10,1909,955,22.765844,99.864377
1,15,15,33091,16546,42.447683,97.649084
2,20,20,207771,103886,51.662139,85.239123
3,25,25,614855,307428,60.667960,56.318260
4,30,30,1043755,521878,65.263244,25.847501


In [ ]:
### 12A.13.5 — Forest-Scale Computational Feasibility

The training-only depth assessment showed that limiting maximum tree depth to
20 substantially reduced individual-tree complexity while retaining greater
structural flexibility than the more restrictive depth limits of 10 and 15.

At max_depth = 20, the single-tree node count was 207,771, representing an
85.24% reduction relative to the unrestricted tree. Increasing max_depth from
20 to 25 increased the node count from 207,771 to 614,855, while a depth limit
of 30 produced 1,043,755 nodes and therefore approached the complexity of the
unrestricted tree.

Accordingly, max_depth = 20 was selected as a computational complexity
constraint rather than a predictively optimised hyperparameter. No validation
observations or predictive performance metrics were used in this decision.

Before fitting the complete baseline forest, a small forest-scale feasibility
probe is performed using 20 trees, max_depth = 20 and sequential tree
construction. This assessment uses only the Fold 1 training data and is
intended solely to verify that multiple depth-controlled trees can be retained
within the available computational resources.

The validation stages remain completely unused during this feasibility
assessment.

In [34]:
# --------------------------------------------------
# Forest-scale computational feasibility probe
# --------------------------------------------------

print("Forest-Scale Computational Feasibility Probe")
print("=" * 70)

print(f"Training observations: {len(X_train_fold1):,}")
print(f"Predictors: {X_train_fold1.shape[1]}")
print("Number of trees: 20")
print("Maximum depth: 20")
print("Parallel jobs: 1")
print("Validation data used: No")
print()

rf_forest_probe = RandomForestRegressor(
    n_estimators=20,
    criterion="squared_error",
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=1.0,
    bootstrap=True,
    max_samples=None,
    random_state=RANDOM_SEED,
    n_jobs=1
)

start_time = time.perf_counter()

rf_forest_probe.fit(
    X_train_fold1,
    y_train_fold1
)

probe_runtime = time.perf_counter() - start_time

tree_nodes = [
    tree.tree_.node_count
    for tree in rf_forest_probe.estimators_
]

tree_depths = [
    tree.tree_.max_depth
    for tree in rf_forest_probe.estimators_
]

print("Probe completed successfully.")
print(f"Runtime: {probe_runtime:.2f} seconds")
print(f"Runtime: {probe_runtime / 60:.2f} minutes")

print(
    f"Mean node count per tree: "
    f"{np.mean(tree_nodes):,.0f}"
)

print(
    f"Minimum node count: "
    f"{np.min(tree_nodes):,}"
)

print(
    f"Maximum node count: "
    f"{np.max(tree_nodes):,}"
)

print(
    f"Mean achieved depth: "
    f"{np.mean(tree_depths):.2f}"
)

print(
    f"Maximum achieved depth: "
    f"{np.max(tree_depths)}"
)

print(
    f"Total nodes across forest: "
    f"{np.sum(tree_nodes):,}"
)

Forest-Scale Computational Feasibility Probe
Training observations: 1,614,240
Predictors: 20
Number of trees: 20
Maximum depth: 20
Parallel jobs: 1
Validation data used: No

Probe completed successfully.
Runtime: 1022.84 seconds
Runtime: 17.05 minutes
Mean node count per tree: 208,314
Minimum node count: 205,821
Maximum node count: 210,737
Mean achieved depth: 20.00
Maximum achieved depth: 20
Total nodes across forest: 4,166,280


In [ ]:
### 12A.13.6 — Controlled Parallel Execution Assessment

The 20-tree forest-scale feasibility assessment completed successfully using
max_depth = 20 and sequential tree construction. The resulting forest contained
4,166,280 nodes and required approximately 17.05 minutes to fit.

This confirmed that limiting tree depth sufficiently reduced forest memory
requirements. However, sequential construction remained computationally
expensive.

Because the number of parallel jobs controls computational execution rather
than the statistical structure of the fitted Random Forest, a limited
parallelisation assessment was performed. The same 20-tree configuration was
therefore fitted using two parallel jobs.

No validation observations or predictive performance metrics were used in this
assessment. Its sole purpose was to determine whether controlled parallel
execution could reduce training time while remaining within available memory.

In [35]:
# --------------------------------------------------
# Controlled parallel-execution feasibility probe
# --------------------------------------------------

print("Controlled Parallel-Execution Feasibility Probe")
print("=" * 70)

print(f"Training observations: {len(X_train_fold1):,}")
print(f"Predictors: {X_train_fold1.shape[1]}")
print("Number of trees: 20")
print("Maximum depth: 20")
print("Parallel jobs: 2")
print("Validation data used: No")
print()

rf_parallel_probe = RandomForestRegressor(
    n_estimators=20,
    criterion="squared_error",
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=1.0,
    bootstrap=True,
    max_samples=None,
    random_state=RANDOM_SEED,
    n_jobs=2
)

start_time = time.perf_counter()

rf_parallel_probe.fit(
    X_train_fold1,
    y_train_fold1
)

parallel_runtime = time.perf_counter() - start_time

parallel_nodes = [
    tree.tree_.node_count
    for tree in rf_parallel_probe.estimators_
]

parallel_depths = [
    tree.tree_.max_depth
    for tree in rf_parallel_probe.estimators_
]

print("Parallel probe completed successfully.")
print(f"Runtime: {parallel_runtime:.2f} seconds")
print(f"Runtime: {parallel_runtime / 60:.2f} minutes")

print(
    f"Mean node count per tree: "
    f"{np.mean(parallel_nodes):,.0f}"
)

print(
    f"Total nodes across forest: "
    f"{np.sum(parallel_nodes):,}"
)

print(
    f"Mean achieved depth: "
    f"{np.mean(parallel_depths):.2f}"
)

speedup = probe_runtime / parallel_runtime

print(
    f"Speed-up relative to n_jobs=1: "
    f"{speedup:.2f}x"
)

Controlled Parallel-Execution Feasibility Probe
Training observations: 1,614,240
Predictors: 20
Number of trees: 20
Maximum depth: 20
Parallel jobs: 2
Validation data used: No

Parallel probe completed successfully.
Runtime: 492.90 seconds
Runtime: 8.21 minutes
Mean node count per tree: 208,314
Total nodes across forest: 4,166,280
Mean achieved depth: 20.00
Speed-up relative to n_jobs=1: 2.08x


In [ ]:
### 12A.13.7 — Four-Worker Parallel Feasibility Assessment

The two-worker parallel feasibility assessment completed successfully and
reduced training time from 17.05 minutes to 8.21 minutes, corresponding to a
2.08-fold speed-up relative to sequential execution.

Importantly, the resulting forest retained the same structural characteristics:
the mean node count, total node count and achieved tree depth were unchanged.
This confirms that controlled parallelisation altered computational execution
rather than the statistical specification of the Random Forest.

Because subsequent chronological model evaluation and hyperparameter tuning
require repeated model fitting, a final controlled parallelisation assessment
is performed using four workers. The statistical Random Forest configuration
remains unchanged.

This assessment again uses only the Fold 1 training data. No validation
observations or predictive performance metrics are used. If four-worker
execution exceeds the available computational resources, the previously
verified two-worker setting will be retained.

In [36]:
# --------------------------------------------------
# Four-worker parallel-execution feasibility probe
# --------------------------------------------------

print("Four-Worker Parallel-Execution Feasibility Probe")
print("=" * 70)

print(f"Training observations: {len(X_train_fold1):,}")
print(f"Predictors: {X_train_fold1.shape[1]}")
print("Number of trees: 20")
print("Maximum depth: 20")
print("Parallel jobs: 4")
print("Validation data used: No")
print()

rf_parallel_probe_4 = RandomForestRegressor(
    n_estimators=20,
    criterion="squared_error",
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=1.0,
    bootstrap=True,
    max_samples=None,
    random_state=RANDOM_SEED,
    n_jobs=4
)

start_time = time.perf_counter()

rf_parallel_probe_4.fit(
    X_train_fold1,
    y_train_fold1
)

parallel_runtime_4 = time.perf_counter() - start_time

parallel_nodes_4 = [
    tree.tree_.node_count
    for tree in rf_parallel_probe_4.estimators_
]

parallel_depths_4 = [
    tree.tree_.max_depth
    for tree in rf_parallel_probe_4.estimators_
]

print("Four-worker probe completed successfully.")
print(f"Runtime: {parallel_runtime_4:.2f} seconds")
print(f"Runtime: {parallel_runtime_4 / 60:.2f} minutes")

print(
    f"Mean node count per tree: "
    f"{np.mean(parallel_nodes_4):,.0f}"
)

print(
    f"Total nodes across forest: "
    f"{np.sum(parallel_nodes_4):,}"
)

print(
    f"Mean achieved depth: "
    f"{np.mean(parallel_depths_4):.2f}"
)

speedup_vs_1 = probe_runtime / parallel_runtime_4
speedup_vs_2 = parallel_runtime / parallel_runtime_4

print(
    f"Speed-up relative to n_jobs=1: "
    f"{speedup_vs_1:.2f}x"
)

print(
    f"Speed-up relative to n_jobs=2: "
    f"{speedup_vs_2:.2f}x"
)

Four-Worker Parallel-Execution Feasibility Probe
Training observations: 1,614,240
Predictors: 20
Number of trees: 20
Maximum depth: 20
Parallel jobs: 4
Validation data used: No

Four-worker probe completed successfully.
Runtime: 225.82 seconds
Runtime: 3.76 minutes
Mean node count per tree: 208,314
Total nodes across forest: 4,166,280
Mean achieved depth: 20.00
Speed-up relative to n_jobs=1: 4.53x
Speed-up relative to n_jobs=2: 2.18x


In [ ]:
### Controlled Parallel Execution Assessment — Summary and Decision

Following the introduction of `max_depth = 20`, the Random Forest became
computationally feasible at forest scale; however, sequential construction
(`n_jobs = 1`) remained time-consuming. A 20-tree feasibility forest required
17.05 minutes to train on the complete Fold 1 training dataset.

Because `n_jobs` controls the number of trees constructed concurrently rather
than the statistical specification of the Random Forest, controlled
parallelisation was assessed using `n_jobs = 1`, `2`, and `4`. The model
configuration, training data, number of trees, and maximum depth were held
constant throughout. No validation observations or predictive performance
metrics were used.

| Setting | n_jobs = 1 | n_jobs = 2 | n_jobs = 4 |
|---|---:|---:|---:|
| Trees | 20 | 20 | 20 |
| max_depth | 20 | 20 | 20 |
| Total nodes | 4,166,280 | 4,166,280 | 4,166,280 |
| Mean nodes/tree | 208,314 | 208,314 | 208,314 |
| Mean achieved depth | 20.00 | 20.00 | 20.00 |
| Runtime | 17.05 min | 8.21 min | 3.76 min |
| Speed-up vs n_jobs = 1 | 1.00× | 2.08× | 4.53× |

The identical forest structural characteristics across all three settings
confirm that parallelisation did not alter the statistical model. Increasing
parallel execution from one to four workers reduced training time from
17.05 minutes to 3.76 minutes, corresponding to a 4.53-fold speed-up.

Therefore, `n_jobs = 4` was selected as the execution setting for subsequent
Random Forest modelling. Further increases in parallelism were not investigated,
because four-worker execution already provided substantial computational
improvement while completing successfully within the available resources.
This decision was based solely on computational feasibility and efficiency,
not predictive performance.

In [39]:
# --------------------------------------------------
# Controlled parallel-execution assessment summary
# --------------------------------------------------

parallel_execution_summary = pd.DataFrame({
    "Setting": [
        "Trees",
        "max_depth",
        "Total nodes",
        "Mean nodes/tree",
        "Mean achieved depth",
        "Runtime (min)",
        "Speed-up vs n_jobs=1"
    ],

    "n_jobs = 1": [
        20,
        20,
        int(np.sum(tree_nodes)),
        int(round(np.mean(tree_nodes))),
        round(np.mean(tree_depths), 2),
        round(probe_runtime / 60, 2),
        1.00
    ],

    "n_jobs = 2": [
        20,
        20,
        int(np.sum(parallel_nodes)),
        int(round(np.mean(parallel_nodes))),
        round(np.mean(parallel_depths), 2),
        round(parallel_runtime / 60, 2),
        round(probe_runtime / parallel_runtime, 2)
    ],

    "n_jobs = 4": [
        20,
        20,
        int(np.sum(parallel_nodes_4)),
        int(round(np.mean(parallel_nodes_4))),
        round(np.mean(parallel_depths_4), 2),
        round(parallel_runtime_4 / 60, 2),
        round(probe_runtime / parallel_runtime_4, 2)
    ]
})

print("Controlled Parallel-Execution Assessment Summary")
print("=" * 70)

display(parallel_execution_summary)

Controlled Parallel-Execution Assessment Summary


,Setting,n_jobs = 1,n_jobs = 2,n_jobs = 4
0,Trees,20.00,20.00,20.00
1,max_depth,20.00,20.00,20.00
2,Total nodes,4166280.00,4166280.00,4166280.00
3,Mean nodes/tree,208314.00,208314.00,208314.00
4,Mean achieved depth,20.00,20.00,20.00
5,Runtime (min),17.05,8.21,3.76
6,Speed-up vs n_jobs=1,1.00,2.08,4.53


In [ ]:
The comparison shows that increasing parallel execution reduced Random Forest
training time substantially while leaving the forest structure unchanged.
Total node count, mean node count per tree and achieved depth were identical
for `n_jobs = 1`, `2` and `4`. Runtime decreased from 17.05 minutes with one
worker to 8.21 minutes with two workers and 3.76 minutes with four workers.
Accordingly, `n_jobs = 4` was retained as the computational execution setting
for subsequent Random Forest modelling. This decision concerns computational
efficiency only and does not constitute predictive hyperparameter optimisation.

In [ ]:
### 12A.13.8 — Final Computationally Constrained Baseline Definition

The initial standard Random Forest baseline permitted unrestricted tree growth
(max_depth = None). On the complete Fold 1 training dataset, a single
unrestricted tree reached a depth of 62 and contained 1,407,579 nodes. At
forest scale, parallel construction of the unrestricted baseline exceeded the
available system memory.

A training-only computational assessment was therefore conducted without using
outer-validation observations or predictive performance metrics. Maximum tree
depths of 10, 15, 20, 25 and 30 were examined. A depth limit of 20 reduced the
single-tree node count to 207,771, corresponding to an 85.24% reduction
relative to unrestricted growth. Increasing the depth limit from 20 to 25
increased the node count to 614,855, while a depth limit of 30 produced
1,043,755 nodes. This demonstrated rapid structural expansion beyond depth 20.

Accordingly, max_depth = 20 was adopted as a computational complexity
constraint. This value was not selected using validation RMSE, MAE or R² and
therefore does not represent predictive hyperparameter optimisation.

Forest-scale feasibility was subsequently verified using 20 trees at
max_depth = 20. Sequential execution completed successfully but required
17.05 minutes. Controlled parallelisation was therefore assessed while holding
the statistical model configuration constant. Two-worker execution reduced
runtime to 8.21 minutes, while four-worker execution reduced runtime further to
3.76 minutes, corresponding to a 4.53-fold speed-up relative to sequential
execution. Forest structural characteristics remained unchanged.

The computationally constrained Random Forest baseline therefore retains the
standard statistical settings wherever possible, with max_depth = 20 introduced
to control demonstrated excessive tree growth and n_jobs = 4 adopted solely as
an execution setting. Other statistical parameters remain unchanged so that
further predictive optimisation is reserved for the subsequent inner
chronological tuning procedure.

In [24]:
# --------------------------------------------------
# Final computationally constrained RF baseline
# --------------------------------------------------

rf_constrained_baseline_config = {
    "n_estimators": 100,
    "criterion": "squared_error",
    "max_depth": 20,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "bootstrap": True,
    "max_samples": None,
    "random_state": RANDOM_SEED,
    "n_jobs": 4
}

print("Final Computationally Constrained RF Baseline")
print("=" * 60)

for parameter, value in rf_constrained_baseline_config.items():
    print(f"{parameter}: {value}")

Final Computationally Constrained RF Baseline
n_estimators: 100
criterion: squared_error
max_depth: 20
min_samples_split: 2
min_samples_leaf: 1
max_features: 1.0
bootstrap: True
max_samples: None
random_state: 42
n_jobs: 4


In [ ]:
## 12A.14 — Chronological Evaluation of the Constrained Random Forest Baseline

Following the computational feasibility assessment, the Random Forest baseline
configuration was frozen before examining predictive performance on the outer
validation stages.

The constrained baseline uses 100 trees with `max_depth = 20`. The depth
constraint was introduced solely to prevent the excessive tree growth observed
during the computational feasibility assessment, while `n_jobs = 4` was
selected as an execution setting to improve computational efficiency. No
outer-validation predictive metrics were used to determine either setting.

The frozen baseline is now evaluated using the same four expanding
chronological outer folds used previously for Ridge Regression and XGBoost:

- Fold 1: train 50–450 h; validate 500–550 h
- Fold 2: train 50–550 h; validate 600–650 h
- Fold 3: train 50–650 h; validate 700–750 h
- Fold 4: train 50–750 h; validate 800–850 h

All observations within the corresponding durability stages are retained, and
the same 20 selected predictors and stack-voltage target are used throughout.

For each fold, predictive performance is quantified using RMSE, MAE and R².
Training and prediction runtimes are also recorded to characterise the
computational cost of the Random Forest baseline.

These outer-fold results provide the pre-tuning Random Forest reference against
which the subsequently tuned Random Forest configuration will be evaluated.
The later-stage 900, 950 and 1000 h holdout observations remain excluded from
this stage.

In [23]:
# --------------------------------------------------
# Create Fold 1 validation data
# using the verified chronological stage definition
# --------------------------------------------------

fold1_val_stages = [500, 550]

# Use the existing development predictor order
fold1_predictors = X_development.columns.tolist()

# Select Fold 1 validation observations
val_mask_fold1 = df["operating_hour"].isin(fold1_val_stages)

X_val_fold1 = df.loc[
    val_mask_fold1,
    fold1_predictors
].copy()

y_val_fold1 = df.loc[
    val_mask_fold1,
    "voltage"
].copy()

print("Fold 1 validation data created successfully.")
print("=" * 65)

print(f"Validation stages: {fold1_val_stages}")
print(f"Number of predictors: {len(fold1_predictors)}")
print()

print(f"X_train_fold1: {X_train_fold1.shape}")
print(f"y_train_fold1: {y_train_fold1.shape}")
print(f"X_val_fold1:   {X_val_fold1.shape}")
print(f"y_val_fold1:   {y_val_fold1.shape}")

print()
print(
    "Predictor order matches development data:",
    X_val_fold1.columns.tolist() == X_development.columns.tolist()
)

Fold 1 validation data created successfully.
Validation stages: [500, 550]
Number of predictors: 20



NameError: name 'X_train_fold1' is not defined

In [45]:
# --------------------------------------------------
# Constrained RF baseline — Outer Fold 1
# --------------------------------------------------

print("Random Forest Constrained Baseline — Outer Fold 1")
print("=" * 70)

print("Training stages: 50–450 h")
print("Validation stages: 500–550 h")
print(f"Training observations: {len(X_train_fold1):,}")
print(f"Validation observations: {len(X_val_fold1):,}")
print(f"Predictors: {X_train_fold1.shape[1]}")
print()

rf_baseline_fold1 = RandomForestRegressor(
    **rf_constrained_baseline_config
)

# -----------------------------
# Training
# -----------------------------

train_start = time.perf_counter()

rf_baseline_fold1.fit(
    X_train_fold1,
    y_train_fold1
)

train_runtime = time.perf_counter() - train_start


# -----------------------------
# Prediction
# -----------------------------

prediction_start = time.perf_counter()

y_pred_fold1 = rf_baseline_fold1.predict(
    X_val_fold1
)

prediction_runtime = (
    time.perf_counter() - prediction_start
)


# -----------------------------
# Evaluation
# -----------------------------

rmse_fold1 = np.sqrt(
    mean_squared_error(
        y_val_fold1,
        y_pred_fold1
    )
)

mae_fold1 = mean_absolute_error(
    y_val_fold1,
    y_pred_fold1
)

r2_fold1 = r2_score(
    y_val_fold1,
    y_pred_fold1
)


# -----------------------------
# Forest complexity
# -----------------------------

fold1_nodes = [
    tree.tree_.node_count
    for tree in rf_baseline_fold1.estimators_
]

fold1_depths = [
    tree.tree_.max_depth
    for tree in rf_baseline_fold1.estimators_
]


# -----------------------------
# Results
# -----------------------------

print("Fold 1 completed successfully.")
print()

print("Predictive Performance")
print("-" * 40)
print(f"RMSE: {rmse_fold1:.6f} V")
print(f"MAE:  {mae_fold1:.6f} V")
print(f"R²:   {r2_fold1:.6f}")

print()

print("Computational Performance")
print("-" * 40)
print(
    f"Training runtime: "
    f"{train_runtime:.2f} seconds "
    f"({train_runtime / 60:.2f} minutes)"
)
print(
    f"Prediction runtime: "
    f"{prediction_runtime:.2f} seconds"
)

print()

print("Forest Structure")
print("-" * 40)
print(
    f"Mean tree depth: "
    f"{np.mean(fold1_depths):.2f}"
)
print(
    f"Mean nodes/tree: "
    f"{np.mean(fold1_nodes):,.0f}"
)
print(
    f"Total forest nodes: "
    f"{np.sum(fold1_nodes):,}"
)

Random Forest Constrained Baseline — Outer Fold 1
Training stages: 50–450 h
Validation stages: 500–550 h
Training observations: 1,614,240
Validation observations: 358,720
Predictors: 20

Fold 1 completed successfully.

Predictive Performance
----------------------------------------
RMSE: 0.008659 V
MAE:  0.006051 V
R²:   0.992160

Computational Performance
----------------------------------------
Training runtime: 1157.03 seconds (19.28 minutes)
Prediction runtime: 1.56 seconds

Forest Structure
----------------------------------------
Mean tree depth: 20.00
Mean nodes/tree: 208,903
Total forest nodes: 20,890,342


In [46]:
# --------------------------------------------------
# Constrained RF baseline — Outer Folds 2–4
# --------------------------------------------------

rf_baseline_results = []

outer_fold_definitions = {
    "Fold_2": {
        "train_stages": list(range(50, 551, 50)),   # 50–550 h
        "val_stages": [600, 650]
    },
    "Fold_3": {
        "train_stages": list(range(50, 651, 50)),   # 50–650 h
        "val_stages": [700, 750]
    },
    "Fold_4": {
        "train_stages": list(range(50, 751, 50)),   # 50–750 h
        "val_stages": [800, 850]
    }
}

predictors = X_development.columns.tolist()

for fold_name, fold_info in outer_fold_definitions.items():

    print("\n" + "=" * 75)
    print(f"Random Forest Constrained Baseline — {fold_name}")
    print("=" * 75)

    train_stages = fold_info["train_stages"]
    val_stages = fold_info["val_stages"]

    # --------------------------------------------------
    # Create train and validation sets
    # --------------------------------------------------

    train_mask = df["operating_hour"].isin(train_stages)
    val_mask = df["operating_hour"].isin(val_stages)

    X_train = df.loc[train_mask, predictors].copy()
    y_train = df.loc[train_mask, "voltage"].copy()

    X_val = df.loc[val_mask, predictors].copy()
    y_val = df.loc[val_mask, "voltage"].copy()

    print(
        f"Training stages: "
        f"{min(train_stages)}–{max(train_stages)} h"
    )
    print(f"Validation stages: {val_stages}")
    print(f"Training observations: {len(X_train):,}")
    print(f"Validation observations: {len(X_val):,}")
    print(f"Predictors: {X_train.shape[1]}")
    print()

    # --------------------------------------------------
    # Fit model
    # --------------------------------------------------

    rf_model = RandomForestRegressor(
        **rf_constrained_baseline_config
    )

    train_start = time.perf_counter()

    rf_model.fit(
        X_train,
        y_train
    )

    train_runtime = (
        time.perf_counter() - train_start
    )

    # --------------------------------------------------
    # Prediction
    # --------------------------------------------------

    prediction_start = time.perf_counter()

    y_pred = rf_model.predict(
        X_val
    )

    prediction_runtime = (
        time.perf_counter() - prediction_start
    )

    # --------------------------------------------------
    # Metrics
    # --------------------------------------------------

    rmse = np.sqrt(
        mean_squared_error(
            y_val,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y_val,
        y_pred
    )

    r2 = r2_score(
        y_val,
        y_pred
    )

    # --------------------------------------------------
    # Forest structure
    # --------------------------------------------------

    tree_nodes = [
        tree.tree_.node_count
        for tree in rf_model.estimators_
    ]

    tree_depths = [
        tree.tree_.max_depth
        for tree in rf_model.estimators_
    ]

    # --------------------------------------------------
    # Save results
    # --------------------------------------------------

    rf_baseline_results.append({
        "Fold": fold_name,
        "Train_Start_h": min(train_stages),
        "Train_End_h": max(train_stages),
        "Validation_Stages": str(val_stages),
        "Train_Observations": len(X_train),
        "Validation_Observations": len(X_val),
        "RMSE_V": rmse,
        "MAE_V": mae,
        "R2": r2,
        "Training_Runtime_min": train_runtime / 60,
        "Prediction_Runtime_s": prediction_runtime,
        "Mean_Tree_Depth": np.mean(tree_depths),
        "Mean_Nodes_per_Tree": np.mean(tree_nodes),
        "Total_Forest_Nodes": np.sum(tree_nodes)
    })

    # --------------------------------------------------
    # Print fold result
    # --------------------------------------------------

    print("Fold completed successfully.")
    print()

    print("Predictive Performance")
    print("-" * 40)
    print(f"RMSE: {rmse:.6f} V")
    print(f"MAE:  {mae:.6f} V")
    print(f"R²:   {r2:.6f}")

    print()

    print("Computational Performance")
    print("-" * 40)
    print(
        f"Training runtime: "
        f"{train_runtime:.2f} seconds "
        f"({train_runtime / 60:.2f} minutes)"
    )
    print(
        f"Prediction runtime: "
        f"{prediction_runtime:.2f} seconds"
    )

    print()

    print("Forest Structure")
    print("-" * 40)
    print(
        f"Mean tree depth: "
        f"{np.mean(tree_depths):.2f}"
    )
    print(
        f"Mean nodes/tree: "
        f"{np.mean(tree_nodes):,.0f}"
    )
    print(
        f"Total forest nodes: "
        f"{np.sum(tree_nodes):,}"
    )

    # --------------------------------------------------
    # Memory cleanup before next fold
    # --------------------------------------------------

    del X_train
    del y_train
    del X_val
    del y_val
    del y_pred
    del rf_model

    gc.collect()


Random Forest Constrained Baseline — Fold_2
Training stages: 50–550 h
Validation stages: [600, 650]
Training observations: 1,972,960
Validation observations: 358,720
Predictors: 20

Fold completed successfully.

Predictive Performance
----------------------------------------
RMSE: 0.009312 V
MAE:  0.006237 V
R²:   0.991317

Computational Performance
----------------------------------------
Training runtime: 2259.90 seconds (37.67 minutes)
Prediction runtime: 3.00 seconds

Forest Structure
----------------------------------------
Mean tree depth: 20.00
Mean nodes/tree: 243,628
Total forest nodes: 24,362,828

Random Forest Constrained Baseline — Fold_3
Training stages: 50–650 h
Validation stages: [700, 750]
Training observations: 2,331,680
Validation observations: 358,720
Predictors: 20

Fold completed successfully.

Predictive Performance
----------------------------------------
RMSE: 0.011430 V
MAE:  0.009343 V
R²:   0.987500

Computational Performance
--------------------------------

In [47]:
# --------------------------------------------------
# 12A.15.1 RF tuning forest-size stability check
# --------------------------------------------------

print("RF Tuning Forest-Size Stability Check")
print("=" * 75)

# --------------------------------------------------
# Chronological inner split
# --------------------------------------------------

stability_train_stages = list(range(50, 651, 50))   # 50–650 h
stability_val_stages = [700, 750]

predictors = X_development.columns.tolist()

train_mask = df["operating_hour"].isin(stability_train_stages)
val_mask = df["operating_hour"].isin(stability_val_stages)

X_stability_train = df.loc[train_mask, predictors].copy()
y_stability_train = df.loc[train_mask, "voltage"].copy()

X_stability_val = df.loc[val_mask, predictors].copy()
y_stability_val = df.loc[val_mask, "voltage"].copy()

print(
    f"Training stages: "
    f"{min(stability_train_stages)}–{max(stability_train_stages)} h"
)
print(f"Validation stages: {stability_val_stages}")
print(f"Training observations: {len(X_stability_train):,}")
print(f"Validation observations: {len(X_stability_val):,}")
print(f"Predictors: {X_stability_train.shape[1]}")
print()


# --------------------------------------------------
# Representative RF structural configurations
# --------------------------------------------------

representative_configs = {
    "Config_A": {
        "max_depth": 15,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 1.0
    },

    "Config_B": {
        "max_depth": 20,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 1.0
    },

    "Config_C": {
        "max_depth": 20,
        "min_samples_split": 10,
        "min_samples_leaf": 5,
        "max_features": 0.8
    },

    "Config_D": {
        "max_depth": 25,
        "min_samples_split": 10,
        "min_samples_leaf": 5,
        "max_features": 0.6
    }
}

tree_counts = [20, 50, 100]

stability_results = []


# --------------------------------------------------
# Run stability experiment
# --------------------------------------------------

for config_name, config in representative_configs.items():

    for n_trees in tree_counts:

        print(
            f"Running {config_name} "
            f"with {n_trees} trees..."
        )

        rf_model = RandomForestRegressor(
            n_estimators=n_trees,
            criterion="squared_error",
            max_depth=config["max_depth"],
            min_samples_split=config["min_samples_split"],
            min_samples_leaf=config["min_samples_leaf"],
            max_features=config["max_features"],
            bootstrap=True,
            max_samples=None,
            random_state=RANDOM_SEED,
            n_jobs=4
        )

        # Training
        start_time = time.perf_counter()

        rf_model.fit(
            X_stability_train,
            y_stability_train
        )

        train_runtime = (
            time.perf_counter() - start_time
        )

        # Validation prediction
        y_pred = rf_model.predict(
            X_stability_val
        )

        # Metrics
        rmse = np.sqrt(
            mean_squared_error(
                y_stability_val,
                y_pred
            )
        )

        mae = mean_absolute_error(
            y_stability_val,
            y_pred
        )

        r2 = r2_score(
            y_stability_val,
            y_pred
        )

        stability_results.append({
            "Configuration": config_name,
            "Trees": n_trees,
            "max_depth": config["max_depth"],
            "min_samples_split": config["min_samples_split"],
            "min_samples_leaf": config["min_samples_leaf"],
            "max_features": config["max_features"],
            "RMSE_V": rmse,
            "MAE_V": mae,
            "R2": r2,
            "Runtime_min": train_runtime / 60
        })

        print(
            f"  RMSE = {rmse:.6f} V | "
            f"MAE = {mae:.6f} V | "
            f"R² = {r2:.6f} | "
            f"Runtime = {train_runtime / 60:.2f} min"
        )

        del rf_model
        del y_pred
        gc.collect()

        print()


# --------------------------------------------------
# Summary table
# --------------------------------------------------

rf_tree_stability_df = pd.DataFrame(
    stability_results
)

display(
    rf_tree_stability_df[
        [
            "Configuration",
            "Trees",
            "max_depth",
            "min_samples_split",
            "min_samples_leaf",
            "max_features",
            "RMSE_V",
            "MAE_V",
            "R2",
            "Runtime_min"
        ]
    ]
)

RF Tuning Forest-Size Stability Check
Training stages: 50–650 h
Validation stages: [700, 750]
Training observations: 2,331,680
Validation observations: 358,720
Predictors: 20

Running Config_A with 20 trees...
  RMSE = 0.011127 V | MAE = 0.009030 V | R² = 0.988154 | Runtime = 7.17 min

Running Config_A with 50 trees...
  RMSE = 0.011126 V | MAE = 0.009036 V | R² = 0.988154 | Runtime = 16.58 min

Running Config_A with 100 trees...
  RMSE = 0.011123 V | MAE = 0.009034 V | R² = 0.988162 | Runtime = 24.25 min

Running Config_B with 20 trees...
  RMSE = 0.011414 V | MAE = 0.009338 V | R² = 0.987533 | Runtime = 6.01 min

Running Config_B with 50 trees...
  RMSE = 0.011425 V | MAE = 0.009338 V | R² = 0.987510 | Runtime = 14.65 min

Running Config_B with 100 trees...
  RMSE = 0.011430 V | MAE = 0.009343 V | R² = 0.987500 | Runtime = 29.54 min

Running Config_C with 20 trees...
  RMSE = 0.011443 V | MAE = 0.009368 V | R² = 0.987471 | Runtime = 4.71 min

Running Config_C with 50 trees...
  RMSE 

,Configuration,Trees,max_depth,min_samples_split,min_samples_leaf,max_features,RMSE_V,MAE_V,R2,Runtime_min
0,Config_A,20,15,2,1,1.0,0.011127,0.009030,0.988154,7.170972
1,Config_A,50,15,2,1,1.0,0.011126,0.009036,0.988154,16.584807
2,Config_A,100,15,2,1,1.0,0.011123,0.009034,0.988162,24.249645
3,Config_B,20,20,2,1,1.0,0.011414,0.009338,0.987533,6.006954
4,Config_B,50,20,2,1,1.0,0.011425,0.009338,0.987510,14.647414
5,Config_B,100,20,2,1,1.0,0.011430,0.009343,0.987500,29.543519
6,Config_C,20,20,10,5,0.8,0.011443,0.009368,0.987471,4.708476
7,Config_C,50,20,10,5,0.8,0.011386,0.009341,0.987596,11.833604
8,Config_C,100,20,10,5,0.8,0.011374,0.009337,0.987622,36.218062
9,Config_D,20,25,10,5,0.6,0.011476,0.009514,0.987399,6.121821


In [48]:
# --------------------------------------------------
# Ranking stability by tree count
# --------------------------------------------------

ranking_summary = []

for n_trees in tree_counts:

    subset = (
        rf_tree_stability_df[
            rf_tree_stability_df["Trees"] == n_trees
        ]
        .sort_values("RMSE_V")
        .reset_index(drop=True)
    )

    subset["Rank"] = np.arange(
        1,
        len(subset) + 1
    )

    ranking_summary.append(
        subset[
            [
                "Configuration",
                "Trees",
                "RMSE_V",
                "Rank"
            ]
        ]
    )

rf_ranking_stability_df = pd.concat(
    ranking_summary,
    ignore_index=True
)

display(
    rf_ranking_stability_df.sort_values(
        ["Trees", "Rank"]
    )
)

,Configuration,Trees,RMSE_V,Rank
0,Config_A,20,0.011127,1
1,Config_B,20,0.011414,2
2,Config_C,20,0.011443,3
3,Config_D,20,0.011476,4
4,Config_A,50,0.011126,1
5,Config_C,50,0.011386,2
6,Config_B,50,0.011425,3
7,Config_D,50,0.011493,4
8,Config_A,100,0.011123,1
9,Config_C,100,0.011374,2


In [49]:
# --------------------------------------------------
# Compact ranking comparison
# --------------------------------------------------

ranking_pivot = (
    rf_ranking_stability_df
    .pivot(
        index="Configuration",
        columns="Trees",
        values="Rank"
    )
    .rename(
        columns={
            20: "Rank_20_Trees",
            50: "Rank_50_Trees",
            100: "Rank_100_Trees"
        }
    )
)

rmse_pivot = (
    rf_tree_stability_df
    .pivot(
        index="Configuration",
        columns="Trees",
        values="RMSE_V"
    )
    .rename(
        columns={
            20: "RMSE_20_Trees",
            50: "RMSE_50_Trees",
            100: "RMSE_100_Trees"
        }
    )
)

rf_stability_summary = (
    ranking_pivot
    .join(rmse_pivot)
    .reset_index()
)

display(rf_stability_summary)

Trees,Configuration,Rank_20_Trees,Rank_50_Trees,Rank_100_Trees,RMSE_20_Trees,RMSE_50_Trees,RMSE_100_Trees
0,Config_A,1,1,1,0.011127,0.011126,0.011123
1,Config_B,2,3,3,0.011414,0.011425,0.011430
2,Config_C,3,2,2,0.011443,0.011386,0.011374
3,Config_D,4,4,4,0.011476,0.011493,0.011497


In [29]:
# --------------------------------------------------
# 12A.15.2 Freeze RF hyperparameter search space
# --------------------------------------------------

print("Random Forest Hyperparameter Search Design")
print("=" * 75)

# --------------------------------------------------
# Fixed inner-search settings
# --------------------------------------------------

RF_TUNING_TREES = 20
RF_FINAL_TREES = 100
RF_N_CONFIGS = 15

# --------------------------------------------------
# Hyperparameter search space
# --------------------------------------------------

rf_param_space = {
    "max_depth": [10, 15, 20, 25],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": [0.5, 0.7, 0.8, 1.0]
}

# --------------------------------------------------
# Generate 15 reproducible candidate configurations
# ONCE — same candidates used for all four outer folds
# --------------------------------------------------

rf_sampled_configs = list(
    ParameterSampler(
        param_distributions=rf_param_space,
        n_iter=RF_N_CONFIGS,
        random_state=RANDOM_SEED
    )
)

# --------------------------------------------------
# Convert to table for inspection
# --------------------------------------------------

rf_candidate_table = pd.DataFrame(
    [
        {
            "Config_ID": f"RF_{i:02d}",
            "n_estimators_inner": RF_TUNING_TREES,
            "max_depth": config["max_depth"],
            "min_samples_split": config["min_samples_split"],
            "min_samples_leaf": config["min_samples_leaf"],
            "max_features": config["max_features"]
        }
        for i, config in enumerate(
            rf_sampled_configs,
            start=1
        )
    ]
)

print(f"Candidate configurations: {len(rf_sampled_configs)}")
print(f"Trees during inner tuning: {RF_TUNING_TREES}")
print(f"Trees during outer refit:   {RF_FINAL_TREES}")
print(f"Random seed:                {RANDOM_SEED}")
print()

display(rf_candidate_table)

Random Forest Hyperparameter Search Design
Candidate configurations: 15
Trees during inner tuning: 20
Trees during outer refit:   100
Random seed:                42



,Config_ID,n_estimators_inner,max_depth,min_samples_split,min_samples_leaf,max_features
0,RF_01,20,25,2,2,0.8
1,RF_02,20,10,10,2,0.5
2,RF_03,20,15,20,10,0.5
3,RF_04,20,25,10,10,0.5
4,RF_05,20,15,5,2,1.0
5,RF_06,20,20,5,5,1.0
6,RF_07,20,25,10,1,1.0
7,RF_08,20,20,20,2,0.8
8,RF_09,20,10,5,5,0.5
9,RF_10,20,10,10,10,0.7


In [30]:
# --------------------------------------------------
# Verify frozen RF tuning design
# --------------------------------------------------

total_inner_fits = (
    RF_N_CONFIGS
    * 2      # inner chronological splits per outer fold
    * 4      # outer folds
)

print("Frozen RF Tuning Design")
print("=" * 60)
print(f"Configurations:                 {RF_N_CONFIGS}")
print("Inner splits per outer fold:    2")
print("Outer folds:                    4")
print(f"Total planned inner fits:       {total_inner_fits}")
print(f"Trees per inner-search forest:  {RF_TUNING_TREES}")
print(f"Trees per outer-refit forest:   {RF_FINAL_TREES}")
print()
print("Holdout stages used in tuning:  NO")
print("Outer validation used in tuning: NO")

Frozen RF Tuning Design
Configurations:                 15
Inner splits per outer fold:    2
Outer folds:                    4
Total planned inner fits:       120
Trees per inner-search forest:  20
Trees per outer-refit forest:   100

Holdout stages used in tuning:  NO
Outer validation used in tuning: NO


In [28]:
# --------------------------------------------------
# Check RF tuning checkpoints after restart
# --------------------------------------------------

import os
import pandas as pd

checkpoint_dir = "rf_tuning_checkpoints"

inner_checkpoint_path = os.path.join(
    checkpoint_dir,
    "rf_inner_tuning_checkpoint.csv"
)

selected_checkpoint_path = os.path.join(
    checkpoint_dir,
    "rf_selected_configs_checkpoint.csv"
)

outer_checkpoint_path = os.path.join(
    checkpoint_dir,
    "rf_outer_tuned_results_checkpoint.csv"
)

print("RF Checkpoint Recovery Check")
print("=" * 70)

print(
    "Inner checkpoint exists:",
    os.path.exists(inner_checkpoint_path)
)

print(
    "Selected-config checkpoint exists:",
    os.path.exists(selected_checkpoint_path)
)

print(
    "Outer-results checkpoint exists:",
    os.path.exists(outer_checkpoint_path)
)

print()

if os.path.exists(inner_checkpoint_path):
    recovered_inner = pd.read_csv(
        inner_checkpoint_path
    )
    
    print(
        f"Recovered successful inner fits: "
        f"{len(recovered_inner)}"
    )
    
    display(recovered_inner.tail(10))

if os.path.exists(selected_checkpoint_path):
    recovered_selected = pd.read_csv(
        selected_checkpoint_path
    )
    
    print()
    print(
        f"Recovered selected outer-fold configs: "
        f"{len(recovered_selected)}"
    )
    
    display(recovered_selected)

if os.path.exists(outer_checkpoint_path):
    recovered_outer = pd.read_csv(
        outer_checkpoint_path
    )
    
    print()
    print(
        f"Recovered completed outer folds: "
        f"{len(recovered_outer)}"
    )
    
    display(recovered_outer)

RF Checkpoint Recovery Check
Inner checkpoint exists: True
Selected-config checkpoint exists: True
Outer-results checkpoint exists: True

Recovered successful inner fits: 42


,Outer_Fold,Inner_Split,Config_ID,n_estimators,max_depth,min_samples_split,min_samples_leaf,max_features,RMSE_V,MAE_V,R2,Training_Runtime_min,Prediction_Runtime_sec,Attempts_Required
32,Fold_2,Inner_1,RF_03,20,15,20,10,0.5,0.020967,0.015399,0.953822,1.292731,0.289371,1
33,Fold_2,Inner_1,RF_04,20,25,10,10,0.5,0.020983,0.015310,0.953751,1.667071,0.392546,1
34,Fold_2,Inner_1,RF_05,20,15,5,2,1.0,0.021491,0.015551,0.951485,2.439381,0.293048,1
35,Fold_2,Inner_1,RF_06,20,20,5,5,1.0,0.021411,0.015453,0.951841,2.842302,0.347972,1
36,Fold_2,Inner_1,RF_07,20,25,10,1,1.0,0.021331,0.015379,0.952201,3.074177,0.395852,1
37,Fold_2,Inner_1,RF_08,20,20,20,2,0.8,0.021266,0.015411,0.952492,2.235805,0.337504,1
38,Fold_2,Inner_1,RF_09,20,10,5,5,0.5,0.021284,0.015682,0.952412,0.887805,0.204931,1
39,Fold_2,Inner_1,RF_10,20,10,10,10,0.7,0.021523,0.015747,0.951340,1.226949,0.195112,1
40,Fold_2,Inner_1,RF_11,20,20,2,2,1.0,0.021370,0.015437,0.952027,2.786571,0.357828,1
41,Fold_2,Inner_1,RF_12,20,25,10,10,0.7,0.021138,0.015313,0.953062,2.136622,0.392605,1



Recovered selected outer-fold configs: 1


,Outer_Fold,Config_ID,max_depth,min_samples_split,min_samples_leaf,max_features,Mean_Inner_RMSE_V,Mean_Inner_MAE_V,Mean_Inner_R2
0,Fold_1,RF_03,15,20,10,0.5,0.015013,0.010865,0.972335



Recovered completed outer folds: 1


,Outer_Fold,Selected_Config,Train_Start_h,Train_End_h,Validation_Stages,RMSE_V,MAE_V,R2,Training_Runtime_min,Prediction_Runtime_sec,Attempts_Required
0,Fold_1,RF_03,50,450,"[500, 550]",0.007979,0.005566,0.993343,8.430549,1.229933,1


In [31]:
# ======================================================================
# 12A.15.3
# Unattended + Progressive + Resumable Nested Chronological RF Tuning
# ======================================================================

import os
import gc
import time
import traceback
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ----------------------------------------------------------------------
# 1. General settings
# ----------------------------------------------------------------------

print("=" * 90)
print("UNATTENDED NESTED CHRONOLOGICAL RANDOM FOREST TUNING")
print("=" * 90)

predictors = X_development.columns.tolist()

TOTAL_OUTER_FOLDS = 4
INNER_SPLITS_PER_FOLD = 2
TOTAL_CONFIGS = RF_N_CONFIGS          # should already be 15
TOTAL_INNER_FITS = (
    TOTAL_OUTER_FOLDS
    * INNER_SPLITS_PER_FOLD
    * TOTAL_CONFIGS
)

MAX_RETRIES_PER_FIT = 3
RETRY_WAIT_SECONDS = 10

print(f"Frozen candidate configurations : {TOTAL_CONFIGS}")
print(f"Inner splits per outer fold     : {INNER_SPLITS_PER_FOLD}")
print(f"Outer folds                     : {TOTAL_OUTER_FOLDS}")
print(f"Total planned inner fits        : {TOTAL_INNER_FITS}")
print(f"Trees during inner tuning       : {RF_TUNING_TREES}")
print(f"Trees during outer refit        : {RF_FINAL_TREES}")
print(f"Maximum retries per failed fit  : {MAX_RETRIES_PER_FIT}")
print()


# ----------------------------------------------------------------------
# 2. Chronological structure
# ----------------------------------------------------------------------

rf_tuning_structure = {

    "Fold_1": {
        "inner_splits": [
            {
                "inner_name": "Inner_1",
                "train_stages": list(range(50, 251, 50)),
                "val_stages": [300, 350]
            },
            {
                "inner_name": "Inner_2",
                "train_stages": list(range(50, 351, 50)),
                "val_stages": [400, 450]
            }
        ],
        "outer_train_stages": list(range(50, 451, 50)),
        "outer_val_stages": [500, 550]
    },

    "Fold_2": {
        "inner_splits": [
            {
                "inner_name": "Inner_1",
                "train_stages": list(range(50, 351, 50)),
                "val_stages": [400, 450]
            },
            {
                "inner_name": "Inner_2",
                "train_stages": list(range(50, 451, 50)),
                "val_stages": [500, 550]
            }
        ],
        "outer_train_stages": list(range(50, 551, 50)),
        "outer_val_stages": [600, 650]
    },

    "Fold_3": {
        "inner_splits": [
            {
                "inner_name": "Inner_1",
                "train_stages": list(range(50, 451, 50)),
                "val_stages": [500, 550]
            },
            {
                "inner_name": "Inner_2",
                "train_stages": list(range(50, 551, 50)),
                "val_stages": [600, 650]
            }
        ],
        "outer_train_stages": list(range(50, 651, 50)),
        "outer_val_stages": [700, 750]
    },

    "Fold_4": {
        "inner_splits": [
            {
                "inner_name": "Inner_1",
                "train_stages": list(range(50, 551, 50)),
                "val_stages": [600, 650]
            },
            {
                "inner_name": "Inner_2",
                "train_stages": list(range(50, 651, 50)),
                "val_stages": [700, 750]
            }
        ],
        "outer_train_stages": list(range(50, 751, 50)),
        "outer_val_stages": [800, 850]
    }
}


# ----------------------------------------------------------------------
# 3. Checkpoint paths
# ----------------------------------------------------------------------

checkpoint_dir = "rf_tuning_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

inner_checkpoint_path = os.path.join(
    checkpoint_dir,
    "rf_inner_tuning_checkpoint.csv"
)

selected_checkpoint_path = os.path.join(
    checkpoint_dir,
    "rf_selected_configs_checkpoint.csv"
)

outer_checkpoint_path = os.path.join(
    checkpoint_dir,
    "rf_outer_tuned_results_checkpoint.csv"
)

error_log_path = os.path.join(
    checkpoint_dir,
    "rf_tuning_error_log.txt"
)


# ----------------------------------------------------------------------
# 4. Load previous progress if files already exist
# ----------------------------------------------------------------------

if os.path.exists(inner_checkpoint_path):
    rf_inner_results_df = pd.read_csv(
        inner_checkpoint_path
    )
    print(
        f"Existing inner checkpoint found: "
        f"{len(rf_inner_results_df)} completed fits."
    )
else:
    rf_inner_results_df = pd.DataFrame()

if os.path.exists(selected_checkpoint_path):
    rf_selected_configs_df = pd.read_csv(
        selected_checkpoint_path
    )
    print(
        f"Existing selected-config checkpoint found: "
        f"{len(rf_selected_configs_df)} completed outer folds."
    )
else:
    rf_selected_configs_df = pd.DataFrame()

if os.path.exists(outer_checkpoint_path):
    rf_outer_tuned_results_df = pd.read_csv(
        outer_checkpoint_path
    )
    print(
        f"Existing outer-results checkpoint found: "
        f"{len(rf_outer_tuned_results_df)} completed outer folds."
    )
else:
    rf_outer_tuned_results_df = pd.DataFrame()

print()


# ----------------------------------------------------------------------
# 5. Helper: determine whether an inner fit is already complete
# ----------------------------------------------------------------------

def inner_fit_already_completed(
    df_results,
    outer_fold,
    inner_split,
    config_id
):
    if df_results.empty:
        return False

    required_columns = {
        "Outer_Fold",
        "Inner_Split",
        "Config_ID"
    }

    if not required_columns.issubset(df_results.columns):
        return False

    mask = (
        (df_results["Outer_Fold"] == outer_fold)
        &
        (df_results["Inner_Split"] == inner_split)
        &
        (df_results["Config_ID"] == config_id)
    )

    return bool(mask.any())


# ----------------------------------------------------------------------
# 6. Helper: safely save checkpoint
# ----------------------------------------------------------------------

def save_inner_checkpoint(df_results):
    df_results.to_csv(
        inner_checkpoint_path,
        index=False
    )


def save_selected_checkpoint(df_results):
    df_results.to_csv(
        selected_checkpoint_path,
        index=False
    )


def save_outer_checkpoint(df_results):
    df_results.to_csv(
        outer_checkpoint_path,
        index=False
    )


# ----------------------------------------------------------------------
# 7. Overall timer
# ----------------------------------------------------------------------

overall_start = time.perf_counter()


# ======================================================================
# 8. OUTER FOLD LOOP
# ======================================================================

for outer_fold_name, outer_info in rf_tuning_structure.items():

    print("\n" + "=" * 90)
    print(f"{outer_fold_name}")
    print("=" * 90)

    # ------------------------------------------------------------------
    # If complete outer-fold result already exists, skip whole fold
    # ------------------------------------------------------------------

    outer_already_complete = False

    if not rf_outer_tuned_results_df.empty:

        if "Outer_Fold" in rf_outer_tuned_results_df.columns:

            outer_already_complete = bool(
                (
                    rf_outer_tuned_results_df[
                        "Outer_Fold"
                    ]
                    == outer_fold_name
                ).any()
            )

    if outer_already_complete:

        print(
            f"{outer_fold_name} already completed previously."
        )
        print("Skipping directly to next outer fold.")

        continue


    # ==================================================================
    # 8A. INNER SPLIT LOOP
    # ==================================================================

    for inner_info in outer_info["inner_splits"]:

        inner_name = inner_info["inner_name"]
        train_stages = inner_info["train_stages"]
        val_stages = inner_info["val_stages"]

        print()
        print("-" * 90)
        print(
            f"{outer_fold_name} | {inner_name}"
        )
        print(
            f"Training stages   : "
            f"{min(train_stages)}–{max(train_stages)} h"
        )
        print(
            f"Validation stages : {val_stages}"
        )
        print("-" * 90)

        # --------------------------------------------------------------
        # Prepare current inner split once
        # --------------------------------------------------------------

        train_mask = df[
            "operating_hour"
        ].isin(train_stages)

        val_mask = df[
            "operating_hour"
        ].isin(val_stages)

        X_inner_train = df.loc[
            train_mask,
            predictors
        ]

        y_inner_train = df.loc[
            train_mask,
            "voltage"
        ]

        X_inner_val = df.loc[
            val_mask,
            predictors
        ]

        y_inner_val = df.loc[
            val_mask,
            "voltage"
        ]

        print(
            f"Training observations   : "
            f"{len(X_inner_train):,}"
        )

        print(
            f"Validation observations : "
            f"{len(X_inner_val):,}"
        )

        print()


        # ==============================================================
        # 8B. CONFIGURATION LOOP
        # ==============================================================

        for config_number, config in enumerate(
            rf_sampled_configs,
            start=1
        ):

            config_id = f"RF_{config_number:02d}"

            # ----------------------------------------------------------
            # Skip completed fits automatically
            # ----------------------------------------------------------

            if inner_fit_already_completed(
                rf_inner_results_df,
                outer_fold_name,
                inner_name,
                config_id
            ):

                print(
                    f"[SKIP] "
                    f"{outer_fold_name} | "
                    f"{inner_name} | "
                    f"{config_id} "
                    f"already completed."
                )

                continue


            # ----------------------------------------------------------
            # Retry loop
            # ----------------------------------------------------------

            fit_successful = False

            for attempt in range(
                1,
                MAX_RETRIES_PER_FIT + 1
            ):

                try:

                    completed_before = (
                        len(rf_inner_results_df)
                    )

                    fit_number = completed_before + 1

                    print()
                    print(
                        f"[{fit_number:03d}/{TOTAL_INNER_FITS}] "
                        f"{outer_fold_name} | "
                        f"{inner_name} | "
                        f"{config_id}"
                    )

                    print(
                        f"Attempt {attempt}/"
                        f"{MAX_RETRIES_PER_FIT}"
                    )

                    print(
                        f"max_depth={config['max_depth']} | "
                        f"min_split="
                        f"{config['min_samples_split']} | "
                        f"min_leaf="
                        f"{config['min_samples_leaf']} | "
                        f"max_features="
                        f"{config['max_features']}"
                    )

                    # --------------------------------------------------
                    # Construct model
                    # --------------------------------------------------

                    rf_inner_model = (
                        RandomForestRegressor(
                            n_estimators=RF_TUNING_TREES,
                            criterion="squared_error",
                            max_depth=config[
                                "max_depth"
                            ],
                            min_samples_split=config[
                                "min_samples_split"
                            ],
                            min_samples_leaf=config[
                                "min_samples_leaf"
                            ],
                            max_features=config[
                                "max_features"
                            ],
                            bootstrap=True,
                            max_samples=None,
                            random_state=RANDOM_SEED,
                            n_jobs=4
                        )
                    )

                    # --------------------------------------------------
                    # Fit
                    # --------------------------------------------------

                    fit_start = time.perf_counter()

                    rf_inner_model.fit(
                        X_inner_train,
                        y_inner_train
                    )

                    fit_runtime = (
                        time.perf_counter()
                        - fit_start
                    )

                    # --------------------------------------------------
                    # Predict
                    # --------------------------------------------------

                    prediction_start = (
                        time.perf_counter()
                    )

                    y_inner_pred = (
                        rf_inner_model.predict(
                            X_inner_val
                        )
                    )

                    prediction_runtime = (
                        time.perf_counter()
                        - prediction_start
                    )

                    # --------------------------------------------------
                    # Metrics
                    # --------------------------------------------------

                    inner_rmse = np.sqrt(
                        mean_squared_error(
                            y_inner_val,
                            y_inner_pred
                        )
                    )

                    inner_mae = (
                        mean_absolute_error(
                            y_inner_val,
                            y_inner_pred
                        )
                    )

                    inner_r2 = r2_score(
                        y_inner_val,
                        y_inner_pred
                    )

                    # --------------------------------------------------
                    # Save successful result
                    # --------------------------------------------------

                    result_row = pd.DataFrame(
                        [{
                            "Outer_Fold":
                                outer_fold_name,

                            "Inner_Split":
                                inner_name,

                            "Config_ID":
                                config_id,

                            "n_estimators":
                                RF_TUNING_TREES,

                            "max_depth":
                                config[
                                    "max_depth"
                                ],

                            "min_samples_split":
                                config[
                                    "min_samples_split"
                                ],

                            "min_samples_leaf":
                                config[
                                    "min_samples_leaf"
                                ],

                            "max_features":
                                config[
                                    "max_features"
                                ],

                            "RMSE_V":
                                inner_rmse,

                            "MAE_V":
                                inner_mae,

                            "R2":
                                inner_r2,

                            "Training_Runtime_min":
                                fit_runtime / 60,

                            "Prediction_Runtime_sec":
                                prediction_runtime,

                            "Attempts_Required":
                                attempt
                        }]
                    )

                    rf_inner_results_df = pd.concat(
                        [
                            rf_inner_results_df,
                            result_row
                        ],
                        ignore_index=True
                    )

                    # --------------------------------------------------
                    # CHECKPOINT IMMEDIATELY AFTER EVERY SUCCESSFUL FIT
                    # --------------------------------------------------

                    save_inner_checkpoint(
                        rf_inner_results_df
                    )

                    print(
                        f"SUCCESS | "
                        f"RMSE={inner_rmse:.6f} V | "
                        f"MAE={inner_mae:.6f} V | "
                        f"R²={inner_r2:.6f} | "
                        f"Train="
                        f"{fit_runtime/60:.2f} min"
                    )

                    print(
                        "Checkpoint saved."
                    )

                    fit_successful = True

                    # --------------------------------------------------
                    # Memory cleanup
                    # --------------------------------------------------

                    del rf_inner_model
                    del y_inner_pred

                    gc.collect()

                    break


                except Exception as e:

                    print()
                    print(
                        f"WARNING: fit attempt "
                        f"{attempt} failed."
                    )

                    print(
                        f"{type(e).__name__}: {e}"
                    )

                    # --------------------------------------------------
                    # Write error to log file
                    # --------------------------------------------------

                    with open(
                        error_log_path,
                        "a",
                        encoding="utf-8"
                    ) as error_file:

                        error_file.write(
                            "\n"
                            + "=" * 80
                            + "\n"
                        )

                        error_file.write(
                            f"Outer Fold: "
                            f"{outer_fold_name}\n"
                        )

                        error_file.write(
                            f"Inner Split: "
                            f"{inner_name}\n"
                        )

                        error_file.write(
                            f"Config: "
                            f"{config_id}\n"
                        )

                        error_file.write(
                            f"Attempt: "
                            f"{attempt}\n"
                        )

                        error_file.write(
                            traceback.format_exc()
                        )

                    # --------------------------------------------------
                    # Aggressive cleanup
                    # --------------------------------------------------

                    try:
                        del rf_inner_model
                    except:
                        pass

                    try:
                        del y_inner_pred
                    except:
                        pass

                    gc.collect()

                    # --------------------------------------------------
                    # Retry automatically
                    # --------------------------------------------------

                    if attempt < MAX_RETRIES_PER_FIT:

                        print(
                            f"Automatic retry in "
                            f"{RETRY_WAIT_SECONDS} seconds..."
                        )

                        time.sleep(
                            RETRY_WAIT_SECONDS
                        )


            # ----------------------------------------------------------
            # If same fit fails repeatedly, stop safely.
            # Do NOT silently continue with incomplete tuning.
            # ----------------------------------------------------------

            if not fit_successful:

                save_inner_checkpoint(
                    rf_inner_results_df
                )

                raise RuntimeError(
                    f"\n"
                    f"{outer_fold_name} | "
                    f"{inner_name} | "
                    f"{config_id} failed after "
                    f"{MAX_RETRIES_PER_FIT} attempts.\n"
                    f"All successful work has been saved.\n"
                    f"Rerunning this cell will automatically "
                    f"resume from the failed configuration."
                )


        # --------------------------------------------------------------
        # Release current inner split
        # --------------------------------------------------------------

        del X_inner_train
        del y_inner_train
        del X_inner_val
        del y_inner_val
        del train_mask
        del val_mask

        gc.collect()


    # ==================================================================
    # 9. VERIFY ALL 30 INNER FITS FOR CURRENT OUTER FOLD
    # ==================================================================

    current_fold_inner = rf_inner_results_df[
        rf_inner_results_df[
            "Outer_Fold"
        ] == outer_fold_name
    ].copy()

    expected_fold_fits = (
        TOTAL_CONFIGS
        * INNER_SPLITS_PER_FOLD
    )

    actual_fold_fits = len(
        current_fold_inner
    )

    print()
    print(
        f"{outer_fold_name} inner fits completed: "
        f"{actual_fold_fits}/"
        f"{expected_fold_fits}"
    )

    if actual_fold_fits != expected_fold_fits:

        raise RuntimeError(
            f"{outer_fold_name} does not contain "
            f"all expected inner tuning results."
        )


    # ==================================================================
    # 10. AGGREGATE INNER PERFORMANCE
    # ==================================================================

    fold_config_summary = (
        current_fold_inner
        .groupby(
            [
                "Config_ID",
                "max_depth",
                "min_samples_split",
                "min_samples_leaf",
                "max_features"
            ],
            as_index=False
        )
        .agg(
            Mean_Inner_RMSE_V=(
                "RMSE_V",
                "mean"
            ),

            Mean_Inner_MAE_V=(
                "MAE_V",
                "mean"
            ),

            Mean_Inner_R2=(
                "R2",
                "mean"
            ),

            Total_Training_Runtime_min=(
                "Training_Runtime_min",
                "sum"
            )
        )
        .sort_values(
            "Mean_Inner_RMSE_V",
            ascending=True
        )
        .reset_index(drop=True)
    )

    fold_config_summary["Rank"] = (
        np.arange(
            1,
            len(fold_config_summary) + 1
        )
    )

    print()
    print(
        f"{outer_fold_name} — "
        f"INNER SEARCH RANKING"
    )

    print("-" * 90)

    display(
        fold_config_summary[
            [
                "Rank",
                "Config_ID",
                "max_depth",
                "min_samples_split",
                "min_samples_leaf",
                "max_features",
                "Mean_Inner_RMSE_V",
                "Mean_Inner_MAE_V",
                "Mean_Inner_R2"
            ]
        ]
    )


    # ==================================================================
    # 11. SELECT WINNER USING MEAN INNER RMSE ONLY
    # ==================================================================

    best_row = (
        fold_config_summary.iloc[0]
    )

    best_config_id = (
        best_row["Config_ID"]
    )

    best_config_number = int(
        best_config_id.split("_")[1]
    )

    best_config = rf_sampled_configs[
        best_config_number - 1
    ]

    selected_row = pd.DataFrame(
        [{
            "Outer_Fold":
                outer_fold_name,

            "Config_ID":
                best_config_id,

            "max_depth":
                best_config[
                    "max_depth"
                ],

            "min_samples_split":
                best_config[
                    "min_samples_split"
                ],

            "min_samples_leaf":
                best_config[
                    "min_samples_leaf"
                ],

            "max_features":
                best_config[
                    "max_features"
                ],

            "Mean_Inner_RMSE_V":
                best_row[
                    "Mean_Inner_RMSE_V"
                ],

            "Mean_Inner_MAE_V":
                best_row[
                    "Mean_Inner_MAE_V"
                ],

            "Mean_Inner_R2":
                best_row[
                    "Mean_Inner_R2"
                ]
        }]
    )

    # Remove old entry if rerunning partially completed fold
    if not rf_selected_configs_df.empty:

        if "Outer_Fold" in (
            rf_selected_configs_df.columns
        ):

            rf_selected_configs_df = (
                rf_selected_configs_df[
                    rf_selected_configs_df[
                        "Outer_Fold"
                    ] != outer_fold_name
                ]
            )

    rf_selected_configs_df = pd.concat(
        [
            rf_selected_configs_df,
            selected_row
        ],
        ignore_index=True
    )

    save_selected_checkpoint(
        rf_selected_configs_df
    )

    print()
    print(
        f"Selected configuration for "
        f"{outer_fold_name}: "
        f"{best_config_id}"
    )

    print(
        f"Mean inner RMSE: "
        f"{best_row['Mean_Inner_RMSE_V']:.6f} V"
    )


    # ==================================================================
    # 12. AUTOMATIC 100-TREE OUTER REFIT
    # ==================================================================

    outer_train_stages = (
        outer_info[
            "outer_train_stages"
        ]
    )

    outer_val_stages = (
        outer_info[
            "outer_val_stages"
        ]
    )

    print()
    print(
        f"Automatically refitting "
        f"{best_config_id} using "
        f"{RF_FINAL_TREES} trees..."
    )

    outer_train_mask = df[
        "operating_hour"
    ].isin(
        outer_train_stages
    )

    outer_val_mask = df[
        "operating_hour"
    ].isin(
        outer_val_stages
    )

    X_outer_train = df.loc[
        outer_train_mask,
        predictors
    ]

    y_outer_train = df.loc[
        outer_train_mask,
        "voltage"
    ]

    X_outer_val = df.loc[
        outer_val_mask,
        predictors
    ]

    y_outer_val = df.loc[
        outer_val_mask,
        "voltage"
    ]


    # ------------------------------------------------------------------
    # Outer refit also gets automatic retries
    # ------------------------------------------------------------------

    outer_fit_successful = False

    for attempt in range(
        1,
        MAX_RETRIES_PER_FIT + 1
    ):

        try:

            tuned_outer_model = (
                RandomForestRegressor(
                    n_estimators=RF_FINAL_TREES,
                    criterion="squared_error",
                    max_depth=best_config[
                        "max_depth"
                    ],
                    min_samples_split=best_config[
                        "min_samples_split"
                    ],
                    min_samples_leaf=best_config[
                        "min_samples_leaf"
                    ],
                    max_features=best_config[
                        "max_features"
                    ],
                    bootstrap=True,
                    max_samples=None,
                    random_state=RANDOM_SEED,
                    n_jobs=4
                )
            )

            print(
                f"Outer refit attempt "
                f"{attempt}/"
                f"{MAX_RETRIES_PER_FIT}"
            )

            outer_fit_start = (
                time.perf_counter()
            )

            tuned_outer_model.fit(
                X_outer_train,
                y_outer_train
            )

            outer_fit_runtime = (
                time.perf_counter()
                - outer_fit_start
            )

            prediction_start = (
                time.perf_counter()
            )

            outer_pred = (
                tuned_outer_model.predict(
                    X_outer_val
                )
            )

            outer_prediction_runtime = (
                time.perf_counter()
                - prediction_start
            )

            outer_rmse = np.sqrt(
                mean_squared_error(
                    y_outer_val,
                    outer_pred
                )
            )

            outer_mae = (
                mean_absolute_error(
                    y_outer_val,
                    outer_pred
                )
            )

            outer_r2 = r2_score(
                y_outer_val,
                outer_pred
            )

            outer_result_row = (
                pd.DataFrame(
                    [{
                        "Outer_Fold":
                            outer_fold_name,

                        "Selected_Config":
                            best_config_id,

                        "Train_Start_h":
                            min(
                                outer_train_stages
                            ),

                        "Train_End_h":
                            max(
                                outer_train_stages
                            ),

                        "Validation_Stages":
                            str(
                                outer_val_stages
                            ),

                        "RMSE_V":
                            outer_rmse,

                        "MAE_V":
                            outer_mae,

                        "R2":
                            outer_r2,

                        "Training_Runtime_min":
                            outer_fit_runtime
                            / 60,

                        "Prediction_Runtime_sec":
                            outer_prediction_runtime,

                        "Attempts_Required":
                            attempt
                    }]
                )
            )

            # Remove duplicate if this fold existed partially
            if not rf_outer_tuned_results_df.empty:

                if "Outer_Fold" in (
                    rf_outer_tuned_results_df.columns
                ):

                    rf_outer_tuned_results_df = (
                        rf_outer_tuned_results_df[
                            rf_outer_tuned_results_df[
                                "Outer_Fold"
                            ] != outer_fold_name
                        ]
                    )

            rf_outer_tuned_results_df = pd.concat(
                [
                    rf_outer_tuned_results_df,
                    outer_result_row
                ],
                ignore_index=True
            )

            save_outer_checkpoint(
                rf_outer_tuned_results_df
            )

            print()
            print(
                f"{outer_fold_name} "
                f"TUNED OUTER PERFORMANCE"
            )

            print("-" * 60)

            print(
                f"RMSE    : "
                f"{outer_rmse:.6f} V"
            )

            print(
                f"MAE     : "
                f"{outer_mae:.6f} V"
            )

            print(
                f"R²      : "
                f"{outer_r2:.6f}"
            )

            print(
                f"Runtime : "
                f"{outer_fit_runtime/60:.2f} min"
            )

            print(
                "Outer-fold checkpoint saved."
            )

            outer_fit_successful = True

            del tuned_outer_model
            del outer_pred

            gc.collect()

            break


        except Exception as e:

            print()
            print(
                f"WARNING: outer refit attempt "
                f"{attempt} failed."
            )

            print(
                f"{type(e).__name__}: {e}"
            )

            with open(
                error_log_path,
                "a",
                encoding="utf-8"
            ) as error_file:

                error_file.write(
                    "\n"
                    + "=" * 80
                    + "\n"
                )

                error_file.write(
                    f"OUTER REFIT\n"
                )

                error_file.write(
                    f"Outer Fold: "
                    f"{outer_fold_name}\n"
                )

                error_file.write(
                    f"Selected Config: "
                    f"{best_config_id}\n"
                )

                error_file.write(
                    f"Attempt: "
                    f"{attempt}\n"
                )

                error_file.write(
                    traceback.format_exc()
                )

            try:
                del tuned_outer_model
            except:
                pass

            try:
                del outer_pred
            except:
                pass

            gc.collect()

            if attempt < MAX_RETRIES_PER_FIT:

                print(
                    f"Automatic retry in "
                    f"{RETRY_WAIT_SECONDS} seconds..."
                )

                time.sleep(
                    RETRY_WAIT_SECONDS
                )


    if not outer_fit_successful:

        raise RuntimeError(
            f"{outer_fold_name} outer refit "
            f"failed after "
            f"{MAX_RETRIES_PER_FIT} attempts. "
            f"Progress is safely checkpointed."
        )


    # ------------------------------------------------------------------
    # Clean outer-fold data
    # ------------------------------------------------------------------

    del X_outer_train
    del y_outer_train
    del X_outer_val
    del y_outer_val
    del outer_train_mask
    del outer_val_mask

    gc.collect()

    print()
    print(
        f"{outer_fold_name} COMPLETED SUCCESSFULLY."
    )


# ======================================================================
# 13. FINAL VERIFICATION
# ======================================================================

total_runtime = (
    time.perf_counter()
    - overall_start
)

print("\n")
print("=" * 90)
print("RANDOM FOREST NESTED TUNING COMPLETE")
print("=" * 90)

completed_inner_fits = len(
    rf_inner_results_df
)

completed_outer_folds = len(
    rf_outer_tuned_results_df
)

print(
    f"Completed inner fits : "
    f"{completed_inner_fits}/"
    f"{TOTAL_INNER_FITS}"
)

print(
    f"Completed outer folds: "
    f"{completed_outer_folds}/"
    f"{TOTAL_OUTER_FOLDS}"
)

print(
    f"Runtime this session : "
    f"{total_runtime/3600:.2f} hours"
)


# ======================================================================
# 14. FINAL SELECTED CONFIGURATIONS
# ======================================================================

print()
print("=" * 90)
print("SELECTED RF CONFIGURATION BY OUTER FOLD")
print("=" * 90)

display(
    rf_selected_configs_df
)


# ======================================================================
# 15. FINAL TUNED OUTER PERFORMANCE
# ======================================================================

print()
print("=" * 90)
print("TUNED RF OUTER-FOLD PERFORMANCE")
print("=" * 90)

display(
    rf_outer_tuned_results_df
)


# ======================================================================
# 16. CROSS-FOLD SUMMARY
# ======================================================================

rf_tuned_summary = pd.DataFrame({
    "Metric": [
        "Mean RMSE (V)",
        "Mean MAE (V)",
        "Mean R²",
        "SD RMSE (V)",
        "SD MAE (V)",
        "SD R²"
    ],

    "Value": [
        rf_outer_tuned_results_df[
            "RMSE_V"
        ].mean(),

        rf_outer_tuned_results_df[
            "MAE_V"
        ].mean(),

        rf_outer_tuned_results_df[
            "R2"
        ].mean(),

        rf_outer_tuned_results_df[
            "RMSE_V"
        ].std(ddof=1),

        rf_outer_tuned_results_df[
            "MAE_V"
        ].std(ddof=1),

        rf_outer_tuned_results_df[
            "R2"
        ].std(ddof=1)
    ]
})

print()
print("=" * 90)
print("TUNED RF CROSS-FOLD SUMMARY")
print("=" * 90)

display(
    rf_tuned_summary
)


# ======================================================================
# 17. SAVE FINAL SUMMARY
# ======================================================================

summary_path = os.path.join(
    checkpoint_dir,
    "rf_tuned_cross_fold_summary.csv"
)

rf_tuned_summary.to_csv(
    summary_path,
    index=False
)

print()
print("=" * 90)
print("ALL REQUIRED RF TUNING OUTPUTS SAVED SUCCESSFULLY")
print("=" * 90)

print(
    f"Inner results      : "
    f"{inner_checkpoint_path}"
)

print(
    f"Selected configs   : "
    f"{selected_checkpoint_path}"
)

print(
    f"Outer results      : "
    f"{outer_checkpoint_path}"
)

print(
    f"Cross-fold summary : "
    f"{summary_path}"
)

print(
    f"Error log          : "
    f"{error_log_path}"
)

print()
print(
    "No 900 h, 950 h or 1000 h data "
    "were used during tuning."
)

print(
    "Nested chronological RF tuning "
    "finished successfully."
)

UNATTENDED NESTED CHRONOLOGICAL RANDOM FOREST TUNING
Frozen candidate configurations : 15
Inner splits per outer fold     : 2
Outer folds                     : 4
Total planned inner fits        : 120
Trees during inner tuning       : 20
Trees during outer refit        : 100
Maximum retries per failed fit  : 3

Existing inner checkpoint found: 42 completed fits.
Existing selected-config checkpoint found: 1 completed outer folds.
Existing outer-results checkpoint found: 1 completed outer folds.


Fold_1
Fold_1 already completed previously.
Skipping directly to next outer fold.

Fold_2

------------------------------------------------------------------------------------------
Fold_2 | Inner_1
Training stages   : 50–350 h
Validation stages : [400, 450]
------------------------------------------------------------------------------------------
Training observations   : 1,255,520
Validation observations : 358,720

[SKIP] Fold_2 | Inner_1 | RF_01 already completed.
[SKIP] Fold_2 | Inner_1 | RF

,Rank,Config_ID,max_depth,min_samples_split,min_samples_leaf,max_features,Mean_Inner_RMSE_V,Mean_Inner_MAE_V,Mean_Inner_R2
0,1,RF_03,15,20,10,0.5,0.014533,0.010526,0.973480
1,2,RF_02,10,10,2,0.5,0.014584,0.010641,0.973458
2,3,RF_04,25,10,10,0.5,0.014594,0.010509,0.973355
3,4,RF_15,20,2,5,0.5,0.014599,0.010604,0.973177
4,5,RF_14,25,5,5,0.7,0.014700,0.010545,0.972957
5,6,RF_09,10,5,5,0.5,0.014724,0.010685,0.972721
6,7,RF_12,25,10,10,0.7,0.014730,0.010548,0.972909
7,8,RF_06,20,5,5,1.0,0.014868,0.010619,0.972297
8,9,RF_01,25,2,2,0.8,0.014900,0.010751,0.972522
9,10,RF_10,10,10,10,0.7,0.014913,0.010686,0.972066



Selected configuration for Fold_2: RF_03
Mean inner RMSE: 0.014533 V

Automatically refitting RF_03 using 100 trees...
Outer refit attempt 1/3

Fold_2 TUNED OUTER PERFORMANCE
------------------------------------------------------------
RMSE    : 0.009069 V
MAE     : 0.006115 V
R²      : 0.991764
Runtime : 9.89 min
Outer-fold checkpoint saved.

Fold_2 COMPLETED SUCCESSFULLY.

Fold_3

------------------------------------------------------------------------------------------
Fold_3 | Inner_1
Training stages   : 50–450 h
Validation stages : [500, 550]
------------------------------------------------------------------------------------------
Training observations   : 1,614,240
Validation observations : 358,720


[061/120] Fold_3 | Inner_1 | RF_01
Attempt 1/3
max_depth=25 | min_split=2 | min_leaf=2 | max_features=0.8
SUCCESS | RMSE=0.008596 V | MAE=0.006115 V | R²=0.992273 | Train=3.42 min
Checkpoint saved.

[062/120] Fold_3 | Inner_1 | RF_02
Attempt 1/3
max_depth=10 | min_split=10 | min_le

,Rank,Config_ID,max_depth,min_samples_split,min_samples_leaf,max_features,Mean_Inner_RMSE_V,Mean_Inner_MAE_V,Mean_Inner_R2
0,1,RF_15,20,2,5,0.5,0.008534,0.005885,0.992542
1,2,RF_04,25,10,10,0.5,0.008581,0.005900,0.992463
2,3,RF_03,15,20,10,0.5,0.008625,0.005903,0.992378
3,4,RF_14,25,5,5,0.7,0.008630,0.005934,0.992377
4,5,RF_12,25,10,10,0.7,0.008724,0.005948,0.992208
5,6,RF_01,25,2,2,0.8,0.008860,0.006142,0.991968
6,7,RF_02,10,10,2,0.5,0.008882,0.006098,0.991909
7,8,RF_06,20,5,5,1.0,0.008905,0.006047,0.991871
8,9,RF_09,10,5,5,0.5,0.008909,0.006080,0.991849
9,10,RF_08,20,20,2,0.8,0.008957,0.006124,0.991792



Selected configuration for Fold_3: RF_15
Mean inner RMSE: 0.008534 V

Automatically refitting RF_15 using 100 trees...
Outer refit attempt 1/3

Fold_3 TUNED OUTER PERFORMANCE
------------------------------------------------------------
RMSE    : 0.011325 V
MAE     : 0.009355 V
R²      : 0.987727
Runtime : 14.49 min
Outer-fold checkpoint saved.

Fold_3 COMPLETED SUCCESSFULLY.

Fold_4

------------------------------------------------------------------------------------------
Fold_4 | Inner_1
Training stages   : 50–550 h
Validation stages : [600, 650]
------------------------------------------------------------------------------------------
Training observations   : 1,972,960
Validation observations : 358,720


[091/120] Fold_4 | Inner_1 | RF_01
Attempt 1/3
max_depth=25 | min_split=2 | min_leaf=2 | max_features=0.8
SUCCESS | RMSE=0.009124 V | MAE=0.006169 V | R²=0.991663 | Train=4.19 min
Checkpoint saved.

[092/120] Fold_4 | Inner_1 | RF_02
Attempt 1/3
max_depth=10 | min_split=10 | min_l

...


TUNED RF OUTER-FOLD PERFORMANCE


,Outer_Fold,Selected_Config,Train_Start_h,Train_End_h,Validation_Stages,RMSE_V,MAE_V,R2,Training_Runtime_min,Prediction_Runtime_sec,Attempts_Required
0,Fold_1,RF_03,50,450,"[500, 550]",0.007979,0.005566,0.993343,8.430549,1.229933,1
1,Fold_2,RF_03,50,550,"[600, 650]",0.009069,0.006115,0.991764,9.888821,1.188556,1
2,Fold_3,RF_15,50,650,"[700, 750]",0.011325,0.009355,0.987727,14.489405,1.753374,1
3,Fold_4,RF_15,50,750,"[800, 850]",0.010574,0.007959,0.988531,17.253662,1.799687,1



TUNED RF CROSS-FOLD SUMMARY


,Metric,Value
0,Mean RMSE (V),0.009737
1,Mean MAE (V),0.007249
2,Mean R²,0.990341
3,SD RMSE (V),0.001501
4,SD MAE (V),0.001738
5,SD R²,0.002655



ALL REQUIRED RF TUNING OUTPUTS SAVED SUCCESSFULLY
Inner results      : rf_tuning_checkpoints\rf_inner_tuning_checkpoint.csv
Selected configs   : rf_tuning_checkpoints\rf_selected_configs_checkpoint.csv
Outer results      : rf_tuning_checkpoints\rf_outer_tuned_results_checkpoint.csv
Cross-fold summary : rf_tuning_checkpoints\rf_tuned_cross_fold_summary.csv
Error log          : rf_tuning_checkpoints\rf_tuning_error_log.txt

No 900 h, 950 h or 1000 h data were used during tuning.
Nested chronological RF tuning finished successfully.


In [32]:
# ============================================================
# 12A.16.1 Final RF Configuration Selection
# Aggregate UNIQUE chronological inner-validation evidence
# ============================================================

import pandas as pd
import numpy as np

print("=" * 88)
print("FINAL RANDOM FOREST CONFIGURATION SELECTION")
print("=" * 88)

# ------------------------------------------------------------
# 1. Load completed inner-tuning evidence
# ------------------------------------------------------------

inner_results_path = (
    "rf_tuning_checkpoints/"
    "rf_inner_tuning_checkpoint.csv"
)

rf_inner_complete = pd.read_csv(inner_results_path)

print(f"Loaded inner-fit records: {len(rf_inner_complete)}")
print(f"Expected records:         120")

if len(rf_inner_complete) != 120:
    raise ValueError(
        "Expected exactly 120 completed inner fits before "
        "final configuration selection."
    )

# ------------------------------------------------------------
# 2. Identify the chronological validation transition
#
# Repeated transitions occur in adjacent outer folds.
# We collapse those duplicates so that the same chronological
# experiment is not counted twice.
# ------------------------------------------------------------

split_map = {
    ("Fold_1", "Inner_1"): "50-250 -> 300-350",
    ("Fold_1", "Inner_2"): "50-350 -> 400-450",

    ("Fold_2", "Inner_1"): "50-350 -> 400-450",
    ("Fold_2", "Inner_2"): "50-450 -> 500-550",

    ("Fold_3", "Inner_1"): "50-450 -> 500-550",
    ("Fold_3", "Inner_2"): "50-550 -> 600-650",

    ("Fold_4", "Inner_1"): "50-550 -> 600-650",
    ("Fold_4", "Inner_2"): "50-650 -> 700-750",
}

rf_inner_complete["Unique_Chronological_Split"] = [
    split_map[(outer_fold, inner_split)]
    for outer_fold, inner_split in zip(
        rf_inner_complete["Outer_Fold"],
        rf_inner_complete["Inner_Split"]
    )
]

# ------------------------------------------------------------
# 3. Collapse duplicated chronological experiments
#
# With deterministic random_state=42, duplicated experiments
# should have identical metrics. Mean is used defensibly in
# case of tiny numerical/runtime variation.
# ------------------------------------------------------------

unique_split_results = (
    rf_inner_complete
    .groupby(
        [
            "Config_ID",
            "max_depth",
            "min_samples_split",
            "min_samples_leaf",
            "max_features",
            "Unique_Chronological_Split"
        ],
        as_index=False
    )
    .agg(
        RMSE_V=("RMSE_V", "mean"),
        MAE_V=("MAE_V", "mean"),
        R2=("R2", "mean")
    )
)

n_unique_splits = (
    unique_split_results["Unique_Chronological_Split"]
    .nunique()
)

print()
print(
    "Unique chronological validation transitions:",
    n_unique_splits
)

print(
    "Expected configuration × unique-split records:",
    15 * n_unique_splits
)

print(
    "Observed configuration × unique-split records:",
    len(unique_split_results)
)

# ------------------------------------------------------------
# 4. Aggregate performance across UNIQUE chronological splits
#
# Primary selection criterion:
# lowest mean RMSE
#
# Secondary metrics are descriptive only.
# ------------------------------------------------------------

rf_final_config_ranking = (
    unique_split_results
    .groupby(
        [
            "Config_ID",
            "max_depth",
            "min_samples_split",
            "min_samples_leaf",
            "max_features"
        ],
        as_index=False
    )
    .agg(
        Mean_RMSE_V=("RMSE_V", "mean"),
        SD_RMSE_V=("RMSE_V", "std"),
        Mean_MAE_V=("MAE_V", "mean"),
        Mean_R2=("R2", "mean"),
        Unique_Splits=("Unique_Chronological_Split", "nunique")
    )
    .sort_values(
        by=["Mean_RMSE_V", "SD_RMSE_V"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

rf_final_config_ranking.insert(
    0,
    "Rank",
    np.arange(1, len(rf_final_config_ranking) + 1)
)

print()
print("=" * 88)
print("FINAL RF CONFIGURATION RANKING")
print("Primary criterion: mean RMSE across unique chronological splits")
print("=" * 88)

display(rf_final_config_ranking)

# ------------------------------------------------------------
# 5. Select final configuration
# ------------------------------------------------------------

rf_final_selection = rf_final_config_ranking.iloc[0].copy()

print()
print("=" * 88)
print("SELECTED FINAL RANDOM FOREST CONFIGURATION")
print("=" * 88)

print(f"Config ID          : {rf_final_selection['Config_ID']}")
print(f"max_depth          : {int(rf_final_selection['max_depth'])}")
print(
    f"min_samples_split  : "
    f"{int(rf_final_selection['min_samples_split'])}"
)
print(
    f"min_samples_leaf   : "
    f"{int(rf_final_selection['min_samples_leaf'])}"
)
print(
    f"max_features       : "
    f"{rf_final_selection['max_features']}"
)

print()
print(
    f"Mean unique-split RMSE : "
    f"{rf_final_selection['Mean_RMSE_V']:.6f} V"
)

print(
    f"SD unique-split RMSE   : "
    f"{rf_final_selection['SD_RMSE_V']:.6f} V"
)

print(
    f"Mean unique-split MAE  : "
    f"{rf_final_selection['Mean_MAE_V']:.6f} V"
)

print(
    f"Mean unique-split R²   : "
    f"{rf_final_selection['Mean_R2']:.6f}"
)

# ------------------------------------------------------------
# 6. Save final ranking
# ------------------------------------------------------------

final_ranking_path = (
    "rf_tuning_checkpoints/"
    "rf_final_configuration_ranking.csv"
)

rf_final_config_ranking.to_csv(
    final_ranking_path,
    index=False
)

print()
print("Final configuration ranking saved:")
print(final_ranking_path)

print()
print(
    "IMPORTANT: No 900 h, 950 h or 1000 h holdout observations "
    "were used in this selection."
)

FINAL RANDOM FOREST CONFIGURATION SELECTION
Loaded inner-fit records: 120
Expected records:         120

Unique chronological validation transitions: 5
Expected configuration × unique-split records: 75
Observed configuration × unique-split records: 75

FINAL RF CONFIGURATION RANKING
Primary criterion: mean RMSE across unique chronological splits


,Rank,Config_ID,max_depth,min_samples_split,min_samples_leaf,max_features,Mean_RMSE_V,SD_RMSE_V,Mean_MAE_V,Mean_R2,Unique_Splits
0,1,RF_03,15,20,10,0.5,0.011715,0.005302,0.008559,0.983442,5
1,2,RF_15,20,2,5,0.5,0.011772,0.005358,0.008626,0.983247,5
2,3,RF_04,25,10,10,0.5,0.011811,0.005277,0.008643,0.983229,5
3,4,RF_14,25,5,5,0.7,0.011857,0.005329,0.008641,0.983066,5
4,5,RF_12,25,10,10,0.7,0.011884,0.005309,0.008641,0.983023,5
5,6,RF_02,10,10,2,0.5,0.011915,0.005182,0.008718,0.983070,5
6,7,RF_09,10,5,5,0.5,0.011973,0.005348,0.008732,0.982777,5
7,8,RF_01,25,2,2,0.8,0.011988,0.005276,0.008761,0.982791,5
8,9,RF_05,15,5,2,1.0,0.011988,0.005398,0.008607,0.982666,5
9,10,RF_08,20,20,2,0.8,0.011998,0.005281,0.008705,0.982754,5



SELECTED FINAL RANDOM FOREST CONFIGURATION
Config ID          : RF_03
max_depth          : 15
min_samples_split  : 20
min_samples_leaf   : 10
max_features       : 0.5

Mean unique-split RMSE : 0.011715 V
SD unique-split RMSE   : 0.005302 V
Mean unique-split MAE  : 0.008559 V
Mean unique-split R²   : 0.983442

Final configuration ranking saved:
rf_tuning_checkpoints/rf_final_configuration_ranking.csv

IMPORTANT: No 900 h, 950 h or 1000 h holdout observations were used in this selection.


In [33]:
# ============================================================
# 12A.16.2 Fit Final Random Forest on Full Development Data
# ============================================================

import time
import gc
import joblib
from sklearn.ensemble import RandomForestRegressor

print("=" * 88)
print("FINAL RANDOM FOREST DEVELOPMENT FIT")
print("=" * 88)

# ------------------------------------------------------------
# 1. Freeze final configuration selected exclusively from
#    development-only inner chronological validation
# ------------------------------------------------------------

final_rf_config = {
    "n_estimators": 100,
    "criterion": "squared_error",
    "max_depth": 15,
    "min_samples_split": 20,
    "min_samples_leaf": 10,
    "max_features": 0.5,
    "bootstrap": True,
    "max_samples": None,
    "random_state": RANDOM_SEED,
    "n_jobs": 4
}

print("Selected configuration: RF_03")
print()
for parameter, value in final_rf_config.items():
    print(f"{parameter:<20}: {value}")

# ------------------------------------------------------------
# 2. Verify final development data
# ------------------------------------------------------------

final_predictors = X_development.columns.tolist()

X_final_train = X_development
y_final_train = development_df["voltage"]

development_stages = sorted(
    development_df["operating_hour"].unique().tolist()
)

print()
print("=" * 88)
print("FINAL DEVELOPMENT DATA")
print("=" * 88)

print(
    f"Development stages : "
    f"{development_stages[0]}–{development_stages[-1]} h"
)

print(
    f"Number of stages   : "
    f"{len(development_stages)}"
)

print(
    f"Training observations: "
    f"{len(X_final_train):,}"
)

print(
    f"Number of predictors: "
    f"{len(final_predictors)}"
)

print(
    "Predictor order matches frozen development matrix:",
    final_predictors == X_development.columns.tolist()
)

# Explicit leakage check
forbidden_stages = {900, 950, 1000}

development_stage_set = set(development_stages)

if development_stage_set.intersection(forbidden_stages):
    raise ValueError(
        "Holdout leakage detected in final development training data."
    )

print(
    "900/950/1000 h present in training data:",
    bool(development_stage_set.intersection(forbidden_stages))
)

# ------------------------------------------------------------
# 3. Fit final RF
# ------------------------------------------------------------

gc.collect()

final_rf_model = RandomForestRegressor(
    **final_rf_config
)

print()
print("=" * 88)
print("TRAINING FINAL 100-TREE RANDOM FOREST")
print("=" * 88)

fit_start = time.time()

final_rf_model.fit(
    X_final_train,
    y_final_train
)

final_rf_training_runtime_min = (
    time.time() - fit_start
) / 60

print("Final RF training completed successfully.")
print(
    f"Training runtime: "
    f"{final_rf_training_runtime_min:.2f} min"
)

# ------------------------------------------------------------
# 4. Verify fitted forest structure
# ------------------------------------------------------------

tree_depths = [
    estimator.tree_.max_depth
    for estimator in final_rf_model.estimators_
]

tree_nodes = [
    estimator.tree_.node_count
    for estimator in final_rf_model.estimators_
]

print()
print("=" * 88)
print("FITTED FOREST VERIFICATION")
print("=" * 88)

print(
    f"Number of fitted trees : "
    f"{len(final_rf_model.estimators_)}"
)

print(
    f"Mean tree depth        : "
    f"{sum(tree_depths) / len(tree_depths):.2f}"
)

print(
    f"Maximum tree depth     : "
    f"{max(tree_depths)}"
)

print(
    f"Mean nodes per tree    : "
    f"{sum(tree_nodes) / len(tree_nodes):,.0f}"
)

print(
    f"Total forest nodes     : "
    f"{sum(tree_nodes):,}"
)

# ------------------------------------------------------------
# 5. Save final fitted model
# ------------------------------------------------------------

final_rf_model_path = (
    "rf_tuning_checkpoints/"
    "final_random_forest_RF03.joblib"
)

joblib.dump(
    final_rf_model,
    final_rf_model_path
)

print()
print("=" * 88)
print("FINAL RF MODEL SAVED")
print("=" * 88)

print(final_rf_model_path)

print()
print(
    "The final RF was trained exclusively on 50–850 h "
    "development data."
)

print(
    "The 900 h, 950 h and 1000 h holdout stages remain "
    "unevaluated by this model."
)

FINAL RANDOM FOREST DEVELOPMENT FIT
Selected configuration: RF_03

n_estimators        : 100
criterion           : squared_error
max_depth           : 15
min_samples_split   : 20
min_samples_leaf    : 10
max_features        : 0.5
bootstrap           : True
max_samples         : None
random_state        : 42
n_jobs              : 4

FINAL DEVELOPMENT DATA
Development stages : 50–850 h
Number of stages   : 17
Training observations: 3,049,120
Number of predictors: 20
Predictor order matches frozen development matrix: True
900/950/1000 h present in training data: False

TRAINING FINAL 100-TREE RANDOM FOREST
Final RF training completed successfully.
Training runtime: 16.38 min

FITTED FOREST VERIFICATION
Number of fitted trees : 100
Mean tree depth        : 15.00
Maximum tree depth     : 15
Mean nodes per tree    : 19,204
Total forest nodes     : 1,920,394

FINAL RF MODEL SAVED
rf_tuning_checkpoints/final_random_forest_RF03.joblib

The final RF was trained exclusively on 50–850 h developmen

In [34]:
# ============================================================
# 12A.17 Final Later-Stage Holdout Evaluation
# Random Forest RF_03
# ============================================================

import time
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

print("=" * 88)
print("FINAL RANDOM FOREST LATER-STAGE HOLDOUT EVALUATION")
print("=" * 88)

# ------------------------------------------------------------
# 1. Verify holdout data
# ------------------------------------------------------------

X_final_holdout = X_holdout
y_final_holdout = holdout_df["voltage"]

holdout_stages = sorted(
    holdout_df["operating_hour"].unique().tolist()
)

expected_holdout_stages = [900, 950, 1000]

print(f"Holdout stages       : {holdout_stages}")
print(f"Holdout observations : {len(X_final_holdout):,}")
print(f"Number of predictors : {X_final_holdout.shape[1]}")

if holdout_stages != expected_holdout_stages:
    raise ValueError(
        "Unexpected holdout stages detected."
    )

if X_final_holdout.columns.tolist() != final_predictors:
    raise ValueError(
        "Holdout predictor order does not match final RF training predictors."
    )

print(
    "Predictor order matches final RF training data:",
    X_final_holdout.columns.tolist() == final_predictors
)

# ------------------------------------------------------------
# 2. Generate final holdout predictions
# ------------------------------------------------------------

print()
print("Generating final RF holdout predictions...")

prediction_start = time.time()

rf_holdout_predictions = final_rf_model.predict(
    X_final_holdout
)

rf_holdout_prediction_runtime_sec = (
    time.time() - prediction_start
)

print("Prediction completed successfully.")
print(
    f"Prediction runtime: "
    f"{rf_holdout_prediction_runtime_sec:.2f} sec"
)

# ------------------------------------------------------------
# 3. Overall holdout performance
# ------------------------------------------------------------

rf_holdout_rmse = np.sqrt(
    mean_squared_error(
        y_final_holdout,
        rf_holdout_predictions
    )
)

rf_holdout_mae = mean_absolute_error(
    y_final_holdout,
    rf_holdout_predictions
)

rf_holdout_r2 = r2_score(
    y_final_holdout,
    rf_holdout_predictions
)

print()
print("=" * 88)
print("OVERALL FINAL RF HOLDOUT PERFORMANCE")
print("=" * 88)

print(f"RMSE : {rf_holdout_rmse:.6f} V")
print(f"     : {rf_holdout_rmse * 1000:.3f} mV")

print(f"MAE  : {rf_holdout_mae:.6f} V")
print(f"     : {rf_holdout_mae * 1000:.3f} mV")

print(f"R²   : {rf_holdout_r2:.6f}")

# ------------------------------------------------------------
# 4. Stage-wise holdout performance
# ------------------------------------------------------------

stage_results = []

holdout_stage_array = (
    holdout_df["operating_hour"]
    .to_numpy()
)

y_holdout_array = y_final_holdout.to_numpy()

for stage in expected_holdout_stages:

    stage_mask = holdout_stage_array == stage

    y_stage = y_holdout_array[stage_mask]

    pred_stage = rf_holdout_predictions[stage_mask]

    stage_rmse = np.sqrt(
        mean_squared_error(
            y_stage,
            pred_stage
        )
    )

    stage_mae = mean_absolute_error(
        y_stage,
        pred_stage
    )

    stage_r2 = r2_score(
        y_stage,
        pred_stage
    )

    stage_results.append(
        {
            "Operating_Hour": stage,
            "Observations": int(stage_mask.sum()),
            "RMSE_V": stage_rmse,
            "RMSE_mV": stage_rmse * 1000,
            "MAE_V": stage_mae,
            "MAE_mV": stage_mae * 1000,
            "R2": stage_r2
        }
    )

rf_stage_holdout_results = pd.DataFrame(
    stage_results
)

print()
print("=" * 88)
print("STAGE-WISE FINAL RF HOLDOUT PERFORMANCE")
print("=" * 88)

display(rf_stage_holdout_results)

# ------------------------------------------------------------
# 5. Save results
# ------------------------------------------------------------

overall_results = pd.DataFrame(
    [
        {
            "Model": "Random Forest",
            "Configuration": "RF_03",
            "Training_Stages": "50-850",
            "Holdout_Stages": "900,950,1000",
            "RMSE_V": rf_holdout_rmse,
            "RMSE_mV": rf_holdout_rmse * 1000,
            "MAE_V": rf_holdout_mae,
            "MAE_mV": rf_holdout_mae * 1000,
            "R2": rf_holdout_r2,
            "Prediction_Runtime_sec":
                rf_holdout_prediction_runtime_sec
        }
    ]
)

overall_path = (
    "rf_tuning_checkpoints/"
    "rf_final_holdout_overall.csv"
)

stage_path = (
    "rf_tuning_checkpoints/"
    "rf_final_holdout_stagewise.csv"
)

overall_results.to_csv(
    overall_path,
    index=False
)

rf_stage_holdout_results.to_csv(
    stage_path,
    index=False
)

print()
print("=" * 88)
print("FINAL RF HOLDOUT RESULTS SAVED")
print("=" * 88)

print(f"Overall results   : {overall_path}")
print(f"Stage-wise results: {stage_path}")

print()
print(
    "Final Random Forest evaluation on the untouched "
    "900–1000 h later-stage holdout is complete."
)

FINAL RANDOM FOREST LATER-STAGE HOLDOUT EVALUATION
Holdout stages       : [900, 950, 1000]
Holdout observations : 580,560
Number of predictors : 20
Predictor order matches final RF training data: True

Generating final RF holdout predictions...
Prediction completed successfully.
Prediction runtime: 2.19 sec

OVERALL FINAL RF HOLDOUT PERFORMANCE
RMSE : 0.008403 V
     : 8.403 mV
MAE  : 0.006195 V
     : 6.195 mV
R²   : 0.992535

STAGE-WISE FINAL RF HOLDOUT PERFORMANCE


,Operating_Hour,Observations,RMSE_V,RMSE_mV,MAE_V,MAE_mV,R2
0,900,179360,0.005531,5.531485,0.004346,4.345667,0.996816
1,950,179360,0.007906,7.905686,0.006004,6.003561,0.993275
2,1000,221840,0.010464,10.464295,0.007845,7.844557,0.988430



FINAL RF HOLDOUT RESULTS SAVED
Overall results   : rf_tuning_checkpoints/rf_final_holdout_overall.csv
Stage-wise results: rf_tuning_checkpoints/rf_final_holdout_stagewise.csv

Final Random Forest evaluation on the untouched 900–1000 h later-stage holdout is complete.


In [35]:
# ============================================================
# 12A.18 Final Comparative Model Assessment
# Ridge vs Random Forest vs XGBoost
# ============================================================

import pandas as pd

print("=" * 88)
print("FINAL COMPARATIVE MODEL ASSESSMENT")
print("=" * 88)

comparison_results = pd.DataFrame(
    [
        {
            "Model": "Ridge Regression",
            "Outer_Mean_RMSE_mV": 17.102,
            "Outer_Mean_MAE_mV": 13.666,
            "Outer_Mean_R2": 0.970199,
            "Holdout_RMSE_mV": 10.694,
            "Holdout_MAE_mV": 7.834,
            "Holdout_R2": 0.987909
        },
        {
            "Model": "XGBoost",
            "Outer_Mean_RMSE_mV": 9.638,
            "Outer_Mean_MAE_mV": 7.155,
            "Outer_Mean_R2": 0.990487,
            "Holdout_RMSE_mV": 8.788,
            "Holdout_MAE_mV": 6.357,
            "Holdout_R2": 0.991834
        },
        {
            "Model": "Random Forest",
            "Outer_Mean_RMSE_mV": 9.737,
            "Outer_Mean_MAE_mV": 7.249,
            "Outer_Mean_R2": 0.990341,
            "Holdout_RMSE_mV": 8.403,
            "Holdout_MAE_mV": 6.195,
            "Holdout_R2": 0.992535
        }
    ]
)

comparison_results["Outer_RMSE_Rank"] = (
    comparison_results["Outer_Mean_RMSE_mV"]
    .rank(method="min")
    .astype(int)
)

comparison_results["Holdout_RMSE_Rank"] = (
    comparison_results["Holdout_RMSE_mV"]
    .rank(method="min")
    .astype(int)
)

comparison_results["Outer_MAE_Rank"] = (
    comparison_results["Outer_Mean_MAE_mV"]
    .rank(method="min")
    .astype(int)
)

comparison_results["Holdout_MAE_Rank"] = (
    comparison_results["Holdout_MAE_mV"]
    .rank(method="min")
    .astype(int)
)

comparison_results["Outer_R2_Rank"] = (
    comparison_results["Outer_Mean_R2"]
    .rank(method="min", ascending=False)
    .astype(int)
)

comparison_results["Holdout_R2_Rank"] = (
    comparison_results["Holdout_R2"]
    .rank(method="min", ascending=False)
    .astype(int)
)

display(comparison_results)

print()
print("=" * 88)
print("KEY COMPARATIVE FINDINGS")
print("=" * 88)

print(
    "1. Ridge Regression was clearly weaker than both nonlinear tree-based models."
)

print(
    "2. XGBoost achieved the lowest mean chronological outer-validation RMSE "
    "(9.638 mV)."
)

print(
    "3. Random Forest achieved the lowest final later-stage holdout RMSE "
    "(8.403 mV)."
)

print(
    "4. XGBoost and Random Forest were closely competitive, with only "
    "0.099 mV separating their mean outer-validation RMSE."
)

print(
    "5. Random Forest outperformed XGBoost on the final holdout by "
    "0.385 mV RMSE."
)

print()
print(
    "Model selection for explainability should be based on a pre-defined "
    "development-stage selection principle rather than selecting whichever "
    "model performs best on the final holdout."
)

FINAL COMPARATIVE MODEL ASSESSMENT


,Model,Outer_Mean_RMSE_mV,Outer_Mean_MAE_mV,Outer_Mean_R2,Holdout_RMSE_mV,Holdout_MAE_mV,Holdout_R2,Outer_RMSE_Rank,Holdout_RMSE_Rank,Outer_MAE_Rank,Holdout_MAE_Rank,Outer_R2_Rank,Holdout_R2_Rank
0,Ridge Regression,17.102,13.666,0.970199,10.694,7.834,0.987909,3,3,3,3,3,3
1,XGBoost,9.638,7.155,0.990487,8.788,6.357,0.991834,1,2,1,2,1,2
2,Random Forest,9.737,7.249,0.990341,8.403,6.195,0.992535,2,1,2,1,2,1



KEY COMPARATIVE FINDINGS
1. Ridge Regression was clearly weaker than both nonlinear tree-based models.
2. XGBoost achieved the lowest mean chronological outer-validation RMSE (9.638 mV).
3. Random Forest achieved the lowest final later-stage holdout RMSE (8.403 mV).
4. XGBoost and Random Forest were closely competitive, with only 0.099 mV separating their mean outer-validation RMSE.
5. Random Forest outperformed XGBoost on the final holdout by 0.385 mV RMSE.

Model selection for explainability should be based on a pre-defined development-stage selection principle rather than selecting whichever model performs best on the final holdout.


In [36]:
# ============================================================
# 12A.18 Final Comparative Model Assessment
# Ridge Regression vs XGBoost vs Random Forest
# ============================================================

import pandas as pd
import numpy as np

print("=" * 92)
print("FINAL COMPARATIVE MODEL ASSESSMENT")
print("=" * 92)

# ------------------------------------------------------------
# 1. Consolidate chronological development-validation results
#    and final later-stage holdout results
# ------------------------------------------------------------

model_comparison = pd.DataFrame(
    [
        {
            "Model": "Ridge Regression",
            "Outer_Mean_RMSE_mV": 17.102,
            "Outer_Mean_MAE_mV": 13.666,
            "Outer_Mean_R2": 0.970199,
            "Holdout_RMSE_mV": 10.694,
            "Holdout_MAE_mV": 7.834,
            "Holdout_R2": 0.987909
        },
        {
            "Model": "XGBoost",
            "Outer_Mean_RMSE_mV": 9.638,
            "Outer_Mean_MAE_mV": 7.155,
            "Outer_Mean_R2": 0.990487,
            "Holdout_RMSE_mV": 8.788,
            "Holdout_MAE_mV": 6.357,
            "Holdout_R2": 0.991834
        },
        {
            "Model": "Random Forest",
            "Outer_Mean_RMSE_mV": 9.737,
            "Outer_Mean_MAE_mV": 7.249,
            "Outer_Mean_R2": 0.990341,
            "Holdout_RMSE_mV": 8.403,
            "Holdout_MAE_mV": 6.195,
            "Holdout_R2": 0.992535
        }
    ]
)

# ------------------------------------------------------------
# 2. Rank models separately for the two evaluation roles
# ------------------------------------------------------------

model_comparison["Outer_RMSE_Rank"] = (
    model_comparison["Outer_Mean_RMSE_mV"]
    .rank(method="min")
    .astype(int)
)

model_comparison["Holdout_RMSE_Rank"] = (
    model_comparison["Holdout_RMSE_mV"]
    .rank(method="min")
    .astype(int)
)

print()
print("COMPARATIVE PERFORMANCE")
print("-" * 92)

display(model_comparison)

# ------------------------------------------------------------
# 3. Quantify RF-XGBoost differences
# ------------------------------------------------------------

xgb_outer = model_comparison.loc[
    model_comparison["Model"] == "XGBoost",
    "Outer_Mean_RMSE_mV"
].iloc[0]

rf_outer = model_comparison.loc[
    model_comparison["Model"] == "Random Forest",
    "Outer_Mean_RMSE_mV"
].iloc[0]

xgb_holdout = model_comparison.loc[
    model_comparison["Model"] == "XGBoost",
    "Holdout_RMSE_mV"
].iloc[0]

rf_holdout = model_comparison.loc[
    model_comparison["Model"] == "Random Forest",
    "Holdout_RMSE_mV"
].iloc[0]

outer_difference = rf_outer - xgb_outer
holdout_difference = xgb_holdout - rf_holdout

print()
print("=" * 92)
print("XGBOOST–RANDOM FOREST COMPARISON")
print("=" * 92)

print(
    f"Chronological outer-validation RMSE difference: "
    f"{outer_difference:.3f} mV in favour of XGBoost"
)

print(
    f"Final holdout RMSE difference: "
    f"{holdout_difference:.3f} mV in favour of Random Forest"
)

# ------------------------------------------------------------
# 4. Freeze model-selection decision
# ------------------------------------------------------------

selected_explainability_model = "XGBoost"

print()
print("=" * 92)
print("MODEL SELECTION FOR EXPLAINABILITY")
print("=" * 92)

print(
    "Primary model-selection criterion:"
)
print(
    "Lowest mean chronological outer-validation RMSE "
    "within the development dataset."
)

print()
print(
    f"Selected model for SHAP: "
    f"{selected_explainability_model}"
)

print()
print(
    "Reason: XGBoost achieved the lowest mean chronological "
    "outer-validation RMSE (9.638 mV), compared with "
    "9.737 mV for Random Forest."
)

print()
print(
    "Random Forest achieved the lowest final later-stage "
    "holdout RMSE (8.403 mV), compared with 8.788 mV "
    "for XGBoost."
)

print()
print(
    "The holdout result does not alter model selection because "
    "the 900–1000 h data were reserved for final later-stage "
    "generalisation assessment rather than model development."
)

print()
print(
    "XGBoost and Random Forest are therefore interpreted as "
    "closely competitive nonlinear models; no claim of "
    "statistical equivalence or universal superiority is made."
)

# ------------------------------------------------------------
# 5. Save comparison evidence
# ------------------------------------------------------------

comparison_path = (
    "rf_tuning_checkpoints/"
    "final_ridge_xgb_rf_comparison.csv"
)

model_comparison.to_csv(
    comparison_path,
    index=False
)

print()
print("=" * 92)
print("COMPARISON EVIDENCE SAVED")
print("=" * 92)

print(comparison_path)

FINAL COMPARATIVE MODEL ASSESSMENT

COMPARATIVE PERFORMANCE
--------------------------------------------------------------------------------------------


,Model,Outer_Mean_RMSE_mV,Outer_Mean_MAE_mV,Outer_Mean_R2,Holdout_RMSE_mV,Holdout_MAE_mV,Holdout_R2,Outer_RMSE_Rank,Holdout_RMSE_Rank
0,Ridge Regression,17.102,13.666,0.970199,10.694,7.834,0.987909,3,3
1,XGBoost,9.638,7.155,0.990487,8.788,6.357,0.991834,1,2
2,Random Forest,9.737,7.249,0.990341,8.403,6.195,0.992535,2,1



XGBOOST–RANDOM FOREST COMPARISON
Chronological outer-validation RMSE difference: 0.099 mV in favour of XGBoost
Final holdout RMSE difference: 0.385 mV in favour of Random Forest

MODEL SELECTION FOR EXPLAINABILITY
Primary model-selection criterion:
Lowest mean chronological outer-validation RMSE within the development dataset.

Selected model for SHAP: XGBoost

Reason: XGBoost achieved the lowest mean chronological outer-validation RMSE (9.638 mV), compared with 9.737 mV for Random Forest.

Random Forest achieved the lowest final later-stage holdout RMSE (8.403 mV), compared with 8.788 mV for XGBoost.

The holdout result does not alter model selection because the 900–1000 h data were reserved for final later-stage generalisation assessment rather than model development.

XGBoost and Random Forest are therefore interpreted as closely competitive nonlinear models; no claim of statistical equivalence or universal superiority is made.

COMPARISON EVIDENCE SAVED
rf_tuning_checkpoints/final